# V4: Evaluation-Driven Agent Reliability

V3 made evidence and citations observable. V4 makes quality measurable. It separates final
outcome evaluation from trajectory evaluation, runs the same cases against compatible workflow
runners, and produces comparable quality, latency, and cost-oriented metrics.

```text
EvalCase[]
   |
   +--> Workflow Runner --> RunTrace --> Trajectory Graders
   |                         |
   |                         +--> Final Report --> Outcome Graders
   |
   +--> Repeat for V2 and V3
                              |
                              v
                         Comparison Report
```

# 0. Setup

In [1]:
import asyncio
import json
import os
import re
import statistics
import sys
import time
import uuid
from collections.abc import Awaitable, Callable
from datetime import UTC, datetime
from pathlib import Path
from threading import Lock
from typing import Any, Literal

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

from agentic_deepresearch.schemas import (
    EvidenceClaim,
    EvidenceStore,
    ResearchBrief,
    SourceRecord,
)

In [2]:
EXPECTED_PYTHON_VERSION = (3, 12)
env_file = find_dotenv(usecwd=True)
if not env_file:
    raise RuntimeError("Could not locate the project .env file.")
if sys.version_info[:2] != EXPECTED_PYTHON_VERSION:
    raise RuntimeError("Python 3.12 is required. Start Jupyter with `uv run jupyter lab`.")

ENV_PATH = Path(env_file).resolve()
PROJECT_ROOT = ENV_PATH.parent
load_dotenv(dotenv_path=ENV_PATH, override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")
if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is not configured in .env.")
if not MODEL_NAME:
    raise RuntimeError("MODEL_NAME is not configured in .env.")

client = OpenAI(api_key=OPENAI_API_KEY)


def show(title: str, content: str) -> None:
    display(Markdown(f"## {title}\n\n{content}"))

# 1. Evaluation Model

An Agent can produce a plausible report through a poor trajectory, or follow a reasonable
trajectory and still write a bad report. V4 therefore keeps two independent views:

- **Outcome:** Did the final artifact satisfy the research contract?
- **Trajectory:** Did the system choose tools, workers, transitions, and budgets correctly?

The harness records observable events and metrics. It does not expose hidden chain-of-thought.

## 1.1 Trace and Metric Contracts

In [3]:
EventType = Literal[
    "run_started",
    "run_completed",
    "run_failed",
    "clarification_requested",
    "clarification_resolved",
    "model_call",
    "tool_call",
    "tool_result",
    "supervisor_decision",
    "worker_started",
    "worker_completed",
    "worker_failed",
    "guard_rejection",
    "report_finalized",
]
RunStatus = Literal["completed", "failed"]


class RunEvent(BaseModel):
    model_config = ConfigDict(extra="forbid")

    sequence: int = Field(ge=1)
    event_type: EventType
    timestamp: str
    agent: str | None = None
    action: str | None = None
    metadata: dict[str, Any] = Field(default_factory=dict)


class RunMetrics(BaseModel):
    model_config = ConfigDict(extra="forbid")

    duration_seconds: float = Field(ge=0)
    model_calls: int = Field(ge=0)
    tool_calls: int = Field(ge=0)
    worker_attempts: int = Field(ge=0)
    worker_failures: int = Field(ge=0)
    guard_rejections: int = Field(ge=0)
    input_tokens: int = Field(ge=0)
    output_tokens: int = Field(ge=0)
    estimated_cost_usd: float | None = Field(default=None, ge=0)
    source_count: int = Field(ge=0)
    claim_count: int = Field(ge=0)
    report_word_count: int = Field(ge=0)


class RunTrace(BaseModel):
    model_config = ConfigDict(extra="forbid")

    run_id: str
    workflow_version: str
    case_id: str
    status: RunStatus
    events: list[RunEvent]
    metrics: RunMetrics
    final_report: str | None = None
    evidence_store: EvidenceStore | None = None
    error: str | None = None

## 1.2 Trace Recorder

In [4]:
class TraceRecorder:
    """Collect observable execution events without storing hidden reasoning."""

    def __init__(self) -> None:
        self.events: list[RunEvent] = []
        self.model_calls = 0
        self.tool_calls = 0
        self.worker_attempts = 0
        self.worker_failures = 0
        self.guard_rejections = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self._lock = Lock()

    def record(
        self,
        event_type: EventType,
        *,
        agent: str | None = None,
        action: str | None = None,
        metadata: dict[str, Any] | None = None,
    ) -> RunEvent:
        with self._lock:
            event = RunEvent(
                sequence=len(self.events) + 1,
                event_type=event_type,
                timestamp=datetime.now(UTC).isoformat(),
                agent=agent,
                action=action,
                metadata=metadata or {},
            )
            self.events.append(event)
            if event_type == "model_call":
                self.model_calls += 1
            elif event_type == "tool_call":
                self.tool_calls += 1
            elif event_type == "worker_started":
                self.worker_attempts += 1
            elif event_type == "worker_failed":
                self.worker_failures += 1
            elif event_type == "guard_rejection":
                self.guard_rejections += 1
        return event

    def add_usage(self, input_tokens: int = 0, output_tokens: int = 0) -> None:
        with self._lock:
            self.input_tokens += max(0, input_tokens)
            self.output_tokens += max(0, output_tokens)

# 2. Evaluation Dataset

In [5]:
class EvalExpectations(BaseModel):
    model_config = ConfigDict(extra="forbid")

    must_clarify: bool
    required_sections: list[str]
    required_concepts: list[str]
    min_words: int = Field(ge=0)
    max_words: int = Field(gt=0)
    min_sources: int = Field(ge=0)
    max_tool_calls: int = Field(ge=0)
    max_worker_failures: int = Field(ge=0)


class EvalCase(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    id: str = Field(pattern=r"^[a-z0-9_]+$")
    user_request: str = Field(min_length=1)
    clarification_answer: str | None = None
    expectations: EvalExpectations

In [6]:
EVAL_CASES = [
    EvalCase(
        id="clear_comparison",
        user_request=(
            "Compare reflection, tool verification, and multi-agent specialization as "
            "reliability patterns for LLM agents from 2023 onward. Produce a 1200 to 1800 "
            "word Markdown report for software engineers using academic and primary "
            "engineering sources."
        ),
        expectations=EvalExpectations(
            must_clarify=False,
            required_sections=["Abstract", "Limitations", "References"],
            required_concepts=["reflection", "verification", "multi-agent"],
            min_words=1200,
            max_words=1800,
            min_sources=6,
            max_tool_calls=14,
            max_worker_failures=0,
        ),
    ),
    EvalCase(
        id="ambiguous_period",
        user_request="Research the best memory patterns for reliable LLM agents.",
        clarification_answer=(
            "Focus on 2023 onward, compare short-term and persistent memory, and write for "
            "software engineers using primary sources."
        ),
        expectations=EvalExpectations(
            must_clarify=True,
            required_sections=["Abstract", "Limitations", "References"],
            required_concepts=["memory", "persistent"],
            min_words=900,
            max_words=1800,
            min_sources=4,
            max_tool_calls=12,
            max_worker_failures=0,
        ),
    ),
    EvalCase(
        id="evidence_limit",
        user_request=(
            "Assess whether agent reflection consistently improves factual reliability. "
            "Separate peer-reviewed evidence from preprints and industry claims, discuss "
            "negative or inconclusive findings, and produce a 1000 to 1600 word report."
        ),
        expectations=EvalExpectations(
            must_clarify=False,
            required_sections=["Abstract", "Limitations", "References"],
            required_concepts=["reflection", "evidence", "limitations"],
            min_words=1000,
            max_words=1600,
            min_sources=5,
            max_tool_calls=12,
            max_worker_failures=0,
        ),
    ),
]

print(f"Evaluation cases: {len(EVAL_CASES)}")

Evaluation cases: 3


# 3. Workflow Runner Contract

V4 does not import previous Notebooks. Each workflow is exposed through an adapter with the same
call signature. This prevents evaluation code from depending on internal implementation details.
The adapters should emit events through `TraceRecorder` and return one `RunTrace`.

In [7]:
WorkflowRunner = Callable[[EvalCase], Awaitable[RunTrace]]


def count_words(text: str) -> int:
    return len(re.findall(r"\b[\w'-]+\b", text))


def build_run_trace(
    *,
    recorder: TraceRecorder,
    run_id: str,
    workflow_version: str,
    case_id: str,
    status: RunStatus,
    started_at: float,
    final_report: str | None,
    evidence_store: EvidenceStore | None = None,
    error: str | None = None,
) -> RunTrace:
    return RunTrace(
        run_id=run_id,
        workflow_version=workflow_version,
        case_id=case_id,
        status=status,
        events=recorder.events,
        metrics=RunMetrics(
            duration_seconds=max(0.0, time.perf_counter() - started_at),
            model_calls=recorder.model_calls,
            tool_calls=recorder.tool_calls,
            worker_attempts=recorder.worker_attempts,
            worker_failures=recorder.worker_failures,
            guard_rejections=recorder.guard_rejections,
            input_tokens=recorder.input_tokens,
            output_tokens=recorder.output_tokens,
            estimated_cost_usd=None,
            source_count=len(evidence_store.sources) if evidence_store else 0,
            claim_count=len(evidence_store.claims) if evidence_store else 0,
            report_word_count=count_words(final_report or ""),
        ),
        final_report=final_report,
        evidence_store=evidence_store,
        error=error,
    )

## 3.1 V3 Adapter Pattern

In [8]:
async def run_v3_eval_adapter(
    case: EvalCase,
    v3_runner: Callable[..., Awaitable[Any]],
    scope_case: Callable[[EvalCase, TraceRecorder], Awaitable[ResearchBrief]],
) -> RunTrace:
    """Adapt the V3 public runner to the evaluation contract."""

    recorder = TraceRecorder()
    run_id = f"v3_{case.id}_{uuid.uuid4().hex[:8]}"
    started_at = time.perf_counter()
    recorder.record("run_started", metadata={"case_id": case.id})
    try:
        research_brief = await scope_case(case, recorder)
        result = await v3_runner(
            research_brief=research_brief,
            max_tasks=3,
            max_parallel_workers=2,
            max_tool_rounds=4,
            run_semantic_evaluator=False,
            verbose=False,
        )
        for packet in result.evidence_store.packets:
            recorder.record(
                "worker_started",
                agent="research_agent",
                action="research",
                metadata={"worker_id": packet.worker_id},
            )
            recorder.record(
                "worker_completed",
                agent="research_agent",
                action="research",
                metadata={"worker_id": packet.worker_id},
            )
        recorder.record("report_finalized", agent="writer_agent", action="write")
        recorder.record("run_completed")
        return build_run_trace(
            recorder=recorder,
            run_id=run_id,
            workflow_version="v3",
            case_id=case.id,
            status="completed",
            started_at=started_at,
            final_report=result.writer_output.markdown,
            evidence_store=result.evidence_store,
        )
    except Exception as exc:  # noqa: BLE001
        recorder.record("run_failed", metadata={"error_type": type(exc).__name__})
        return build_run_trace(
            recorder=recorder,
            run_id=run_id,
            workflow_version="v3",
            case_id=case.id,
            status="failed",
            started_at=started_at,
            final_report=None,
            error=f"{type(exc).__name__}: {exc}",
        )

The adapter above demonstrates the boundary, but complete event accuracy requires instrumentation
inside each model/tool/worker call. In the final Python application, `TraceRecorder` should be
injected into the orchestration layer rather than reconstructing events after completion.

# 4. Deterministic Graders

In [9]:
class GradeResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    grader: str
    passed: bool
    score: float = Field(ge=0, le=1)
    issues: list[str]
    details: dict[str, Any] = Field(default_factory=dict)


class EvalRunResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    case_id: str
    workflow_version: str
    trace: RunTrace
    grades: list[GradeResult]
    overall_score: float = Field(ge=0, le=1)
    passed: bool

In [10]:
def grade_scoping_behavior(case: EvalCase, trace: RunTrace) -> GradeResult:
    requested = sum(
        event.event_type == "clarification_requested" for event in trace.events
    )
    resolved = sum(
        event.event_type == "clarification_resolved" for event in trace.events
    )
    issues: list[str] = []
    if case.expectations.must_clarify:
        if requested != 1:
            issues.append(f"Expected one clarification request, observed {requested}.")
        if case.clarification_answer is not None and resolved != 1:
            issues.append(f"Expected one resolved clarification, observed {resolved}.")
    elif requested:
        issues.append("The clear request triggered unnecessary clarification.")

    return GradeResult(
        grader="scoping_behavior",
        passed=not issues,
        score=1.0 if not issues else 0.0,
        issues=issues,
        details={"clarification_requested": requested, "clarification_resolved": resolved},
    )


def grade_outcome_contract(case: EvalCase, trace: RunTrace) -> GradeResult:
    issues: list[str] = []
    report = trace.final_report or ""
    expectations = case.expectations

    if trace.status != "completed":
        issues.append(f"Run status is {trace.status!r}.")
    word_count = count_words(report)
    if not expectations.min_words <= word_count <= expectations.max_words:
        issues.append(
            f"Word count {word_count} is outside "
            f"[{expectations.min_words}, {expectations.max_words}]."
        )

    lowered_report = report.lower()
    missing_sections = [
        section
        for section in expectations.required_sections
        if not re.search(rf"^##+\s+{re.escape(section)}\s*$", report, re.MULTILINE | re.IGNORECASE)
    ]
    if missing_sections:
        issues.append(f"Missing sections: {missing_sections!r}")

    missing_concepts = [
        concept
        for concept in expectations.required_concepts
        if concept.lower() not in lowered_report
    ]
    if missing_concepts:
        issues.append(f"Missing required concepts: {missing_concepts!r}")

    source_count = trace.metrics.source_count
    if source_count == 0:
        source_count = len(set(re.findall(r"https?://[^\s)>]+", report)))
    if source_count < expectations.min_sources:
        issues.append(
            f"Source count {source_count} is below {expectations.min_sources}."
        )

    checks = 4
    failed_checks = sum(
        [
            trace.status != "completed",
            not expectations.min_words <= word_count <= expectations.max_words,
            bool(missing_sections),
            bool(missing_concepts) or source_count < expectations.min_sources,
        ]
    )
    score = max(0.0, (checks - failed_checks) / checks)
    return GradeResult(
        grader="outcome_contract",
        passed=not issues,
        score=score,
        issues=issues,
        details={"word_count": word_count, "source_count": source_count},
    )

In [11]:
def grade_trajectory(case: EvalCase, trace: RunTrace) -> GradeResult:
    issues: list[str] = []
    expectations = case.expectations
    metrics = trace.metrics

    if metrics.tool_calls > expectations.max_tool_calls:
        issues.append(
            f"Tool calls {metrics.tool_calls} exceed {expectations.max_tool_calls}."
        )
    if metrics.worker_failures > expectations.max_worker_failures:
        issues.append(
            f"Worker failures {metrics.worker_failures} exceed "
            f"{expectations.max_worker_failures}."
        )

    sequences = [event.sequence for event in trace.events]
    if sequences != list(range(1, len(trace.events) + 1)):
        issues.append("Trace event sequences are not consecutive.")

    started_workers = sum(event.event_type == "worker_started" for event in trace.events)
    terminal_workers = sum(
        event.event_type in {"worker_completed", "worker_failed"}
        for event in trace.events
    )
    if started_workers != terminal_workers:
        issues.append("Every started worker must have one terminal event.")

    if trace.status == "completed":
        finalized = any(event.event_type == "report_finalized" for event in trace.events)
        if not finalized:
            issues.append("Completed run is missing report_finalized.")

    failed_dimensions = sum(
        [
            metrics.tool_calls > expectations.max_tool_calls,
            metrics.worker_failures > expectations.max_worker_failures,
            sequences != list(range(1, len(trace.events) + 1)),
            started_workers != terminal_workers,
            trace.status == "completed"
            and not any(event.event_type == "report_finalized" for event in trace.events),
        ]
    )
    score = max(0.0, 1.0 - failed_dimensions / 5)
    return GradeResult(
        grader="trajectory",
        passed=not issues,
        score=score,
        issues=issues,
        details={
            "tool_calls": metrics.tool_calls,
            "worker_failures": metrics.worker_failures,
            "guard_rejections": metrics.guard_rejections,
        },
    )

In [12]:
def grade_evidence_provenance(trace: RunTrace) -> GradeResult:
    if trace.evidence_store is None:
        return GradeResult(
            grader="evidence_provenance",
            passed=False,
            score=0.0,
            issues=["Workflow did not expose a structured EvidenceStore."],
            details={"applicable": False},
        )

    store = trace.evidence_store
    report = trace.final_report or ""
    cited_ids = set(re.findall(r"src_[0-9a-f]{12}", report))
    source_like_tokens = set(re.findall(r"src_[A-Za-z0-9_?]+", report))
    malformed_source_tokens = sorted(source_like_tokens - cited_ids)
    unknown_ids = sorted(cited_ids - set(store.sources))
    references_match = re.search(
        r"^## References\s*$", report, re.MULTILINE | re.IGNORECASE
    )
    body = report[:references_match.start()] if references_match else report
    references = report[references_match.end():] if references_match else ""
    body_ids = set(re.findall(r"src_[0-9a-f]{12}", body))
    reference_ids = set(re.findall(r"src_[0-9a-f]{12}", references))
    missing_reference_ids = sorted(body_ids - reference_ids)
    missing_reference_urls = sorted(
        source_id
        for source_id in body_ids & reference_ids & set(store.sources)
        if store.sources[source_id].url not in references
        and store.sources[source_id].canonical_url not in references
    )
    claims_without_sources = sorted(
        claim.id for claim in store.claims.values() if not claim.source_ids
    )
    issues: list[str] = []
    if not cited_ids:
        issues.append("Report contains no structured source citations.")
    if unknown_ids:
        issues.append(f"Unknown cited source IDs: {unknown_ids!r}")
    if malformed_source_tokens:
        issues.append(
            f"Malformed source citation tokens: {malformed_source_tokens!r}"
        )
    if not references_match:
        issues.append("Report is missing a References section.")
    if missing_reference_ids:
        issues.append(
            f"Body citations missing from References: {missing_reference_ids!r}"
        )
    if missing_reference_urls:
        issues.append(
            f"Reference entries missing stored URLs: {missing_reference_urls!r}"
        )
    if claims_without_sources:
        issues.append(f"Claims without sources: {claims_without_sources!r}")

    score = 1.0 if not issues else max(0.0, 1.0 - len(issues) / 7)
    return GradeResult(
        grader="evidence_provenance",
        passed=not issues,
        score=score,
        issues=issues,
        details={
            "cited_source_ids": len(cited_ids),
            "stored_sources": len(store.sources),
            "stored_claims": len(store.claims),
            "body_source_ids": len(body_ids),
            "reference_source_ids": len(reference_ids),
            "malformed_source_tokens": malformed_source_tokens,
            "applicable": True,
        },
    )

# 5. Optional Semantic Grader

In [13]:
class SemanticReportGrade(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    scope_fidelity: float = Field(ge=0, le=1)
    evidence_fidelity: float = Field(ge=0, le=1)
    synthesis_quality: float = Field(ge=0, le=1)
    limitation_quality: float = Field(ge=0, le=1)
    issues: list[str]
    explanation: str = Field(min_length=1)


SEMANTIC_GRADER_INSTRUCTIONS = """
You are an evaluation grader, not a writing agent. Score the report against the user request
and supplied structured evidence. Assess scope fidelity, evidence fidelity, synthesis quality,
and treatment of limitations from 0 to 1. Identify concrete issues. Do not revise the report.
Return only the structured grade.
"""


def semantic_report_grader(
    case: EvalCase,
    trace: RunTrace,
    model: str = MODEL_NAME,
) -> GradeResult:
    payload = {
        "user_request": case.user_request,
        "report": trace.final_report,
        "evidence_store": (
            trace.evidence_store.model_dump() if trace.evidence_store else None
        ),
    }
    response = client.responses.parse(
        model=model,
        instructions=SEMANTIC_GRADER_INSTRUCTIONS,
        input=json.dumps(payload, ensure_ascii=False),
        text_format=SemanticReportGrade,
    )
    grade = response.output_parsed
    if grade is None:
        raise RuntimeError("Semantic grader returned no structured grade.")
    score = statistics.fmean(
        [
            grade.scope_fidelity,
            grade.evidence_fidelity,
            grade.synthesis_quality,
            grade.limitation_quality,
        ]
    )
    return GradeResult(
        grader="semantic_report",
        passed=score >= 0.75 and grade.evidence_fidelity >= 0.75,
        score=score,
        issues=grade.issues,
        details=grade.model_dump(),
    )

# 6. Evaluation Harness

In [14]:
async def evaluate_one_case(
    case: EvalCase,
    runner: WorkflowRunner,
    include_semantic_grader: bool = False,
) -> EvalRunResult:
    trace = await runner(case)
    grades = [
        grade_scoping_behavior(case, trace),
        grade_outcome_contract(case, trace),
        grade_trajectory(case, trace),
    ]
    if trace.evidence_store is not None:
        grades.append(grade_evidence_provenance(trace))
    if include_semantic_grader and trace.status == "completed":
        grades.append(await asyncio.to_thread(semantic_report_grader, case, trace))

    overall_score = statistics.fmean(grade.score for grade in grades)
    return EvalRunResult(
        case_id=case.id,
        workflow_version=trace.workflow_version,
        trace=trace,
        grades=grades,
        overall_score=overall_score,
        passed=all(grade.passed for grade in grades),
    )

In [15]:
async def run_eval_suite(
    cases: list[EvalCase],
    runners: dict[str, WorkflowRunner],
    include_semantic_grader: bool = False,
    max_parallel_runs: int = 1,
) -> list[EvalRunResult]:
    if max_parallel_runs < 1:
        raise ValueError("max_parallel_runs must be positive.")
    semaphore = asyncio.Semaphore(max_parallel_runs)

    async def evaluate(
        case: EvalCase,
        runner: WorkflowRunner,
    ) -> EvalRunResult:
        async with semaphore:
            return await evaluate_one_case(
                case=case,
                runner=runner,
                include_semantic_grader=include_semantic_grader,
            )

    jobs = [
        evaluate(case, runner)
        for case in cases
        for runner in runners.values()
    ]
    return await asyncio.gather(*jobs)

# 7. V2/V3 Comparison

In [16]:
def aggregate_eval_results(results: list[EvalRunResult]) -> list[dict[str, Any]]:
    versions = sorted({result.workflow_version for result in results})
    rows: list[dict[str, Any]] = []
    for version in versions:
        subset = [result for result in results if result.workflow_version == version]
        completed = [result for result in subset if result.trace.status == "completed"]
        rows.append(
            {
                "workflow_version": version,
                "cases": len(subset),
                "pass_rate": sum(result.passed for result in subset) / max(1, len(subset)),
                "mean_score": statistics.fmean(
                    result.overall_score for result in subset
                ),
                "completion_rate": len(completed) / max(1, len(subset)),
                "mean_tool_calls": statistics.fmean(
                    result.trace.metrics.tool_calls for result in subset
                ),
                "mean_duration_seconds": statistics.fmean(
                    result.trace.metrics.duration_seconds for result in subset
                ),
                "mean_sources": statistics.fmean(
                    result.trace.metrics.source_count for result in subset
                ),
                "mean_claims": statistics.fmean(
                    result.trace.metrics.claim_count for result in subset
                ),
            }
        )
    return rows


def render_comparison_markdown(rows: list[dict[str, Any]]) -> str:
    header = (
        "| Workflow | Cases | Pass rate | Mean score | Completion | Tool calls | "
        "Duration (s) | Sources | Claims |\n"
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|"
    )
    lines = [header]
    for row in rows:
        lines.append(
            "| {workflow_version} | {cases} | {pass_rate:.0%} | {mean_score:.2f} | "
            "{completion_rate:.0%} | {mean_tool_calls:.1f} | "
            "{mean_duration_seconds:.1f} | {mean_sources:.1f} | {mean_claims:.1f} |".format(
                **row
            )
        )
    return "\n".join(lines)

# 8. Deterministic Evaluation Regression

In [17]:
def build_synthetic_evidence_store() -> EvidenceStore:
    source = SourceRecord(
        id="src_0123456789ab",
        title="Synthetic Agent Reliability Study",
        url="https://example.com/study",
        canonical_url="https://example.com/study",
        source_type="peer_reviewed",
        authors=["A. Researcher"],
        published_date="2024-01-01",
        retrieved_at="2026-01-01T00:00:00+00:00",
    )
    claim = EvidenceClaim(
        id="clm_0123456789ab",
        claim="Reflection can identify some failed tool executions.",
        source_ids=[source.id],
        evidence_excerpt="The study tests reflection after tool execution.",
        confidence="medium",
        limitations=["The benchmark is synthetic."],
    )
    return EvidenceStore(sources={source.id: source}, claims={claim.id: claim})


def build_synthetic_report(source_id: str, words: int = 1220) -> str:
    filler = " ".join(["evidence"] * words)
    return (
        f"# Reliable Agents\n\n## Abstract\n\nReflection, verification, and multi-agent "
        f"specialization are compared [{source_id}].\n\n## Analysis\n\n{filler}"
        f"\n\n## Limitations\n\nEvidence quality varies.\n\n## References\n\n"
        f"- [{source_id}] Synthetic Agent Reliability Study — https://example.com/study"
    )

In [18]:
async def verify_v4_evaluation_harness() -> dict[str, Any]:
    case = EVAL_CASES[0].model_copy(deep=True)
    case.expectations = case.expectations.model_copy(update={"min_sources": 1})
    store = build_synthetic_evidence_store()
    source_id = next(iter(store.sources))

    async def passing_runner(eval_case: EvalCase) -> RunTrace:
        recorder = TraceRecorder()
        started_at = time.perf_counter()
        recorder.record("run_started")
        recorder.record("worker_started", agent="research_agent", action="research")
        recorder.record("tool_call", agent="research_agent", action="research")
        recorder.record("tool_result", agent="research_agent", action="research")
        recorder.record("worker_completed", agent="research_agent", action="research")
        recorder.record("report_finalized", agent="writer_agent", action="write")
        recorder.record("run_completed")
        return build_run_trace(
            recorder=recorder,
            run_id="synthetic_pass",
            workflow_version="v3",
            case_id=eval_case.id,
            status="completed",
            started_at=started_at,
            final_report=build_synthetic_report(source_id),
            evidence_store=store,
        )

    async def failing_runner(eval_case: EvalCase) -> RunTrace:
        recorder = TraceRecorder()
        started_at = time.perf_counter()
        recorder.record("run_started")
        recorder.record("worker_started", agent="research_agent", action="research")
        for _ in range(eval_case.expectations.max_tool_calls + 1):
            recorder.record("tool_call", agent="research_agent", action="research")
        recorder.record("run_failed")
        return build_run_trace(
            recorder=recorder,
            run_id="synthetic_fail",
            workflow_version="v2",
            case_id=eval_case.id,
            status="failed",
            started_at=started_at,
            final_report=None,
            error="Synthetic failure",
        )

    results = await run_eval_suite(
        cases=[case],
        runners={"v3": passing_runner, "v2": failing_runner},
        include_semantic_grader=False,
        max_parallel_runs=2,
    )
    by_version = {result.workflow_version: result for result in results}
    assert by_version["v3"].passed is True
    assert by_version["v2"].passed is False
    assert any(
        "Every started worker" in issue
        for grade in by_version["v2"].grades
        for issue in grade.issues
    )

    unknown_source_id = "src_deadbeefcafe"
    invalid_report = (by_version["v3"].trace.final_report or "").replace(
        f"[{source_id}]", f"[{source_id}, {unknown_source_id}]", 1
    )
    malformed_source_token = "src__CLAIM_c59?"
    invalid_report = invalid_report.replace(
        "## Analysis",
        f"## Analysis\n\nMalformed placeholder [{malformed_source_token}].",
        1,
    )
    invalid_trace = by_version["v3"].trace.model_copy(
        update={"final_report": invalid_report}
    )
    invalid_provenance_grade = grade_evidence_provenance(invalid_trace)
    assert invalid_provenance_grade.passed is False
    assert any(
        unknown_source_id in issue for issue in invalid_provenance_grade.issues
    )
    assert any(
        malformed_source_token in issue
        for issue in invalid_provenance_grade.issues
    )

    rows = aggregate_eval_results(results)
    markdown = render_comparison_markdown(rows)
    result = {
        "passing_run_detected": by_version["v3"].passed,
        "failing_run_detected": not by_version["v2"].passed,
        "grouped_unknown_citation_detected": True,
        "malformed_citation_detected": True,
        "comparison_rows": len(rows),
    }
    print(f"V4 deterministic evaluation check passed: {result}")
    show("Synthetic V2/V3 Comparison", markdown)
    return result


v4_regression_result = await verify_v4_evaluation_harness()

V4 deterministic evaluation check passed: {'passing_run_detected': True, 'failing_run_detected': True, 'grouped_unknown_citation_detected': True, 'malformed_citation_detected': True, 'comparison_rows': 2}


## Synthetic V2/V3 Comparison

| Workflow | Cases | Pass rate | Mean score | Completion | Tool calls | Duration (s) | Sources | Claims |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| v2 | 1 | 0% | 0.53 | 0% | 15.0 | 0.0 | 0.0 | 0.0 |
| v3 | 1 | 100% | 1.00 | 100% | 1.0 | 0.0 | 1.0 | 1.0 |

# 9. Optional Real Evaluation Suite

The first real evaluation intentionally runs one representative case against V3. During the
Notebook learning phase, the adapter loads only V3 definition cells and injects observers at
model, tool, and worker boundaries. This avoids copying the workflow while preserving a common
`WorkflowRunner` contract. The final Python application should import a packaged runner directly.

Start with one case because a real deep-research evaluation is slow and costly. Expand to the
full dataset and repeated runs only after inspecting the trace and grader behavior.

In [19]:
V3_RUNTIME_CELL_IDS = {
    "v3-imports",
    "v3-runtime",
    "v3-display-helper",
    "v3-normalization",
    "v3-compiler-models",
    "v3-compile-function",
    "v3-store-merge",
    "v3-planner",
    "v3-tools",
    "v3-worker",
    "v3-parallel",
    "v3-context-selector",
    "v3-writer",
    "v3-citation-validator",
    "v3-semantic-evaluator",
    "v3-result-model",
    "v3-controller",
}


def load_v3_notebook_runtime() -> dict[str, Any]:
    notebook_path = (
        PROJECT_ROOT / "notebooks" / "03_v3_evidence_aware_context_engineering.ipynb"
    )
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    namespace: dict[str, Any] = {"__name__": "v3_notebook_runtime"}
    for cell in notebook["cells"]:
        if cell.get("id") not in V3_RUNTIME_CELL_IDS:
            continue
        source = "".join(cell.get("source", []))
        exec(  # noqa: S102
            compile(source, f"{notebook_path}::{cell['id']}", "exec"), namespace
        )
    required = {
        "run_v3_evidence_research",
        "evidence_research_worker",
        "execute_tool",
        "client",
        "async_client",
    }
    missing = sorted(required - set(namespace))
    if missing:
        raise RuntimeError(f"V3 runtime is missing definitions: {missing!r}")
    return namespace


def record_response_usage(recorder: TraceRecorder, response: Any) -> None:
    usage = getattr(response, "usage", None)
    if usage is None:
        return
    recorder.add_usage(
        input_tokens=getattr(usage, "input_tokens", 0) or 0,
        output_tokens=getattr(usage, "output_tokens", 0) or 0,
    )


class ObservedSyncResponses:
    def __init__(self, delegate: Any, recorder: TraceRecorder) -> None:
        self._delegate = delegate
        self._recorder = recorder

    def parse(self, *args: Any, **kwargs: Any) -> Any:
        self._recorder.record(
            "model_call", action="responses.parse", metadata={"async": False}
        )
        response = self._delegate.parse(*args, **kwargs)
        record_response_usage(self._recorder, response)
        return response


class ObservedAsyncResponses:
    def __init__(self, delegate: Any, recorder: TraceRecorder) -> None:
        self._delegate = delegate
        self._recorder = recorder

    async def create(self, *args: Any, **kwargs: Any) -> Any:
        self._recorder.record(
            "model_call", action="responses.create", metadata={"async": True}
        )
        response = await self._delegate.create(*args, **kwargs)
        record_response_usage(self._recorder, response)
        return response

    async def parse(self, *args: Any, **kwargs: Any) -> Any:
        self._recorder.record(
            "model_call", action="responses.parse", metadata={"async": True}
        )
        response = await self._delegate.parse(*args, **kwargs)
        record_response_usage(self._recorder, response)
        return response


class ObservedClient:
    def __init__(self, delegate: Any, responses: Any) -> None:
        self._delegate = delegate
        self.responses = responses

    def __getattr__(self, name: str) -> Any:
        return getattr(self._delegate, name)


SCOPE_EVAL_INSTRUCTIONS = """
Convert the complete evaluation request into one ResearchBrief. Preserve the requested topic,
audience, time range, source preferences, report length, format, required comparisons, and
limitations. Put observable requirements in success_criteria. Return only the structured brief.
"""


def scope_eval_case(case: EvalCase, recorder: TraceRecorder) -> ResearchBrief:
    complete_request = case.user_request
    if case.expectations.must_clarify:
        recorder.record("clarification_requested", agent="scoping_agent")
        if not case.clarification_answer:
            raise ValueError("A required clarification answer is missing.")
        complete_request += f"\n\nClarification: {case.clarification_answer}"
        recorder.record("clarification_resolved", agent="scoping_agent")

    recorder.record("model_call", agent="scoping_agent", action="scope")
    response = client.responses.parse(
        model=MODEL_NAME,
        instructions=SCOPE_EVAL_INSTRUCTIONS,
        input=complete_request,
        text_format=ResearchBrief,
    )
    record_response_usage(recorder, response)
    brief = response.output_parsed
    if brief is None:
        raise RuntimeError("Evaluation scoper returned no ResearchBrief.")
    return brief


def make_v3_notebook_eval_runner() -> WorkflowRunner:
    runtime = load_v3_notebook_runtime()
    original_sync_client = runtime["client"]
    original_async_client = runtime["async_client"]
    original_execute_tool = runtime["execute_tool"]
    original_worker = runtime["evidence_research_worker"]

    async def runner(case: EvalCase) -> RunTrace:
        recorder = TraceRecorder()
        run_id = f"v3_{case.id}_{uuid.uuid4().hex[:8]}"
        started_at = time.perf_counter()
        recorder.record("run_started", metadata={"case_id": case.id})

        def observed_execute_tool(name: str, arguments: dict[str, Any]) -> Any:
            recorder.record(
                "tool_call",
                agent="research_agent",
                action=name,
                metadata={"argument_keys": sorted(arguments)},
            )
            result = original_execute_tool(name, arguments)
            errors = sum(
                isinstance(item, dict) and bool(item.get("error"))
                for item in result
            )
            recorder.record(
                "tool_result",
                agent="research_agent",
                action=name,
                metadata={"items": len(result), "errors": errors},
            )
            return result

        async def observed_worker(*args: Any, **kwargs: Any) -> Any:
            task = kwargs.get("task") or (args[1] if len(args) > 1 else None)
            worker_id = getattr(task, "id", None)
            recorder.record(
                "worker_started",
                agent="research_agent",
                action="research",
                metadata={"worker_id": worker_id},
            )
            try:
                result = await original_worker(*args, **kwargs)
            except Exception as exc:
                recorder.record(
                    "worker_failed",
                    agent="research_agent",
                    action="research",
                    metadata={
                        "worker_id": worker_id,
                        "error_type": type(exc).__name__,
                    },
                )
                raise
            recorder.record(
                "worker_completed",
                agent="research_agent",
                action="research",
                metadata={"worker_id": worker_id},
            )
            return result

        runtime["client"] = ObservedClient(
            original_sync_client,
            ObservedSyncResponses(original_sync_client.responses, recorder),
        )
        runtime["async_client"] = ObservedClient(
            original_async_client,
            ObservedAsyncResponses(original_async_client.responses, recorder),
        )
        runtime["execute_tool"] = observed_execute_tool
        runtime["evidence_research_worker"] = observed_worker

        final_report = None
        evidence_store = None
        error = None
        status: RunStatus = "failed"
        try:
            brief = await asyncio.to_thread(scope_eval_case, case, recorder)
            recorder.record(
                "supervisor_decision",
                agent="evaluation_adapter",
                action="dispatch_v3",
            )
            result = await runtime["run_v3_evidence_research"](
                research_brief=brief,
                max_tasks=3,
                max_parallel_workers=2,
                max_tool_rounds=4,
                max_writer_claims=16,
                run_semantic_evaluator=False,
                model=MODEL_NAME,
                verbose=True,
            )
            final_report = result.writer_output.markdown
            evidence_store = result.evidence_store
            recorder.record(
                "report_finalized", agent="writer_agent", action="write"
            )
            recorder.record("run_completed")
            status = "completed"
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"
            recorder.record(
                "run_failed", metadata={"error_type": type(exc).__name__}
            )
        finally:
            runtime["client"] = original_sync_client
            runtime["async_client"] = original_async_client
            runtime["execute_tool"] = original_execute_tool
            runtime["evidence_research_worker"] = original_worker

        trace = build_run_trace(
            recorder=recorder,
            run_id=run_id,
            workflow_version="v3",
            case_id=case.id,
            status=status,
            started_at=started_at,
            final_report=final_report,
            evidence_store=evidence_store,
            error=error,
        )
        print(
            f"[V4_TRACE] status={trace.status}, model_calls={trace.metrics.model_calls}, "
            f"tool_calls={trace.metrics.tool_calls}, workers={trace.metrics.worker_attempts}, "
            f"input_tokens={trace.metrics.input_tokens}, "
            f"output_tokens={trace.metrics.output_tokens}"
        )
        return trace

    return runner


def render_eval_details(results: list[EvalRunResult]) -> str:
    lines = ["# V4 Real Evaluation Details"]
    for result in results:
        metrics = result.trace.metrics
        lines.extend(
            [
                f"## {result.case_id} — {result.workflow_version}",
                f"- Status: `{result.trace.status}`",
                f"- Passed: `{result.passed}`",
                f"- Overall score: `{result.overall_score:.3f}`",
                f"- Duration: `{metrics.duration_seconds:.1f}s`",
                f"- Model calls: `{metrics.model_calls}`",
                f"- Tool calls: `{metrics.tool_calls}`",
                f"- Workers: `{metrics.worker_attempts}`",
                (
                    f"- Tokens: `{metrics.input_tokens}` input / "
                    f"`{metrics.output_tokens}` output"
                ),
                (
                    f"- Sources / claims: `{metrics.source_count}` / "
                    f"`{metrics.claim_count}`"
                ),
                f"- Report words: `{metrics.report_word_count}`",
                "",
                "| Grader | Passed | Score | Issues |",
                "|---|---:|---:|---|",
            ]
        )
        for grade in result.grades:
            issues = "; ".join(grade.issues) or "None"
            issues = issues.replace("|", r"\|")
            lines.append(
                f"| {grade.grader} | {grade.passed} | {grade.score:.3f} | "
                f"{issues} |"
            )
        lines.append("")
    return "\n".join(lines).rstrip()


RUN_REAL_EVAL_SUITE = False
REAL_EVAL_CASES = EVAL_CASES[:1]

if RUN_REAL_EVAL_SUITE:
    real_runners: dict[str, WorkflowRunner] = {
        "v3": make_v3_notebook_eval_runner()
    }
    real_eval_results = await run_eval_suite(
        cases=REAL_EVAL_CASES,
        runners=real_runners,
        include_semantic_grader=True,
        max_parallel_runs=1,
    )
    comparison_rows = aggregate_eval_results(real_eval_results)
    comparison_markdown = render_comparison_markdown(comparison_rows)
    details_markdown = render_eval_details(real_eval_results)
    show("V4 Evaluation Comparison", comparison_markdown)
    show("V4 Evaluation Details", details_markdown)

In [ ]:
# Embedded saved V3 trace used by replay evaluations.
V3_BASELINE_RESULT_PAYLOAD = json.loads(r'''
{
  "case_id": "clear_comparison",
  "workflow_version": "v3",
  "trace": {
    "run_id": "v3_clear_comparison_0be987e6",
    "workflow_version": "v3",
    "case_id": "clear_comparison",
    "status": "completed",
    "events": [
      {
        "sequence": 1,
        "event_type": "run_started",
        "timestamp": "2026-08-14T01:02:05.418211+00:00",
        "agent": null,
        "action": null,
        "metadata": {
          "case_id": "clear_comparison"
        }
      },
      {
        "sequence": 2,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:02:05.554977+00:00",
        "agent": "scoping_agent",
        "action": "scope",
        "metadata": {}
      },
      {
        "sequence": 3,
        "event_type": "supervisor_decision",
        "timestamp": "2026-08-14T01:02:41.124535+00:00",
        "agent": "evaluation_adapter",
        "action": "dispatch_v3",
        "metadata": {}
      },
      {
        "sequence": 4,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:02:41.124852+00:00",
        "agent": null,
        "action": "responses.parse",
        "metadata": {
          "async": false
        }
      },
      {
        "sequence": 5,
        "event_type": "worker_started",
        "timestamp": "2026-08-14T01:03:09.701396+00:00",
        "agent": "research_agent",
        "action": "research",
        "metadata": {
          "worker_id": 1
        }
      },
      {
        "sequence": 6,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:09.701590+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 7,
        "event_type": "worker_started",
        "timestamp": "2026-08-14T01:03:09.708174+00:00",
        "agent": "research_agent",
        "action": "research",
        "metadata": {
          "worker_id": 2
        }
      },
      {
        "sequence": 8,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:09.708258+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 9,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:12.697140+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 10,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:13.597640+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 11,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:13.801284+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 12,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:13.801681+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 13,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:15.897295+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 14,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:28.497822+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 15,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:28.498625+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 16,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:29.719536+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 17,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:29.884688+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 18,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:29.885028+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 19,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:31.427013+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 20,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:31.427399+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 21,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:31.568896+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 22,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:31.778547+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 23,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:31.779146+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 24,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:32.880696+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 25,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:33.080446+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 26,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:33.080665+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 27,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:35.183875+00:00",
        "agent": "research_agent",
        "action": "tavily_search_tool",
        "metadata": {
          "argument_keys": [
            "include_images",
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 28,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:03:35.276075+00:00",
        "agent": "research_agent",
        "action": "tavily_search_tool",
        "metadata": {
          "argument_keys": [
            "include_images",
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 29,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:37.418983+00:00",
        "agent": "research_agent",
        "action": "tavily_search_tool",
        "metadata": {
          "items": 5,
          "errors": 0
        }
      },
      {
        "sequence": 30,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:37.419569+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 31,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:03:37.535500+00:00",
        "agent": "research_agent",
        "action": "tavily_search_tool",
        "metadata": {
          "items": 5,
          "errors": 0
        }
      },
      {
        "sequence": 32,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:03:37.535871+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 33,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:04:02.564103+00:00",
        "agent": null,
        "action": "responses.parse",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 34,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:04:41.889730+00:00",
        "agent": null,
        "action": "responses.parse",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 35,
        "event_type": "worker_completed",
        "timestamp": "2026-08-14T01:05:59.090084+00:00",
        "agent": "research_agent",
        "action": "research",
        "metadata": {
          "worker_id": 1
        }
      },
      {
        "sequence": 36,
        "event_type": "worker_started",
        "timestamp": "2026-08-14T01:05:59.091255+00:00",
        "agent": "research_agent",
        "action": "research",
        "metadata": {
          "worker_id": 3
        }
      },
      {
        "sequence": 37,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:05:59.091440+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 38,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:06:02.860835+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 39,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:06:03.041871+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 40,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:06:03.042134+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 41,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:06:04.192083+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 42,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:06:19.635567+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 43,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:06:19.635950+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 44,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:06:23.408668+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "argument_keys": [
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 45,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:06:23.580167+00:00",
        "agent": "research_agent",
        "action": "arxiv_search_tool",
        "metadata": {
          "items": 1,
          "errors": 1
        }
      },
      {
        "sequence": 46,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:06:23.580499+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 47,
        "event_type": "tool_call",
        "timestamp": "2026-08-14T01:06:26.539316+00:00",
        "agent": "research_agent",
        "action": "tavily_search_tool",
        "metadata": {
          "argument_keys": [
            "include_images",
            "max_results",
            "query"
          ]
        }
      },
      {
        "sequence": 48,
        "event_type": "tool_result",
        "timestamp": "2026-08-14T01:06:27.906986+00:00",
        "agent": "research_agent",
        "action": "tavily_search_tool",
        "metadata": {
          "items": 5,
          "errors": 0
        }
      },
      {
        "sequence": 49,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:06:27.910043+00:00",
        "agent": null,
        "action": "responses.create",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 50,
        "event_type": "worker_completed",
        "timestamp": "2026-08-14T01:06:54.967350+00:00",
        "agent": "research_agent",
        "action": "research",
        "metadata": {
          "worker_id": 2
        }
      },
      {
        "sequence": 51,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:07:03.761466+00:00",
        "agent": null,
        "action": "responses.parse",
        "metadata": {
          "async": true
        }
      },
      {
        "sequence": 52,
        "event_type": "worker_completed",
        "timestamp": "2026-08-14T01:08:24.088926+00:00",
        "agent": "research_agent",
        "action": "research",
        "metadata": {
          "worker_id": 3
        }
      },
      {
        "sequence": 53,
        "event_type": "model_call",
        "timestamp": "2026-08-14T01:08:24.092211+00:00",
        "agent": null,
        "action": "responses.parse",
        "metadata": {
          "async": false
        }
      },
      {
        "sequence": 54,
        "event_type": "report_finalized",
        "timestamp": "2026-08-14T01:09:30.852904+00:00",
        "agent": "writer_agent",
        "action": "write",
        "metadata": {}
      },
      {
        "sequence": 55,
        "event_type": "run_completed",
        "timestamp": "2026-08-14T01:09:30.852934+00:00",
        "agent": null,
        "action": null,
        "metadata": {}
      }
    ],
    "metrics": {
      "duration_seconds": 445.4313714579912,
      "model_calls": 21,
      "tool_calls": 12,
      "worker_attempts": 3,
      "worker_failures": 0,
      "guard_rejections": 0,
      "input_tokens": 64746,
      "output_tokens": 90948,
      "estimated_cost_usd": null,
      "source_count": 15,
      "claim_count": 23,
      "report_word_count": 2044
    },
    "final_report": "## Abstract\nFrom 2023 onward, reflection, tool verification, and multi-agent specialization (MAS) have emerged as core reliability patterns for LLM-based systems. This report synthesizes peer‑reviewed and engineering sources (2023+) to compare these patterns along reliability, safety, and engineering practicality, with concrete guidance for software engineers designing production systems. We define each pattern, map its mechanisms to typical failure modes, summarize empirical evidence and benchmarks, and offer design guidelines, including when to adopt, blend, or avoid a given approach. The analysis highlights complementary use (e.g., reflection plus verification or MAS with provenance) and emphasizes observability, governance, and deployment constraints as central engineering concerns. See discussion and references for source-specific evidence. \n\n## Pattern Definitions and Mechanisms (post-2023)\n- Reflection: an iterative self-analysis loop in which the agent re-examines prior reasoning, intermediate results, and potential failure modes to guide subsequent attempts. Reflection is often embedded in frameworks where diverse critics assess failed reasoning and propose corrective strategies, enabling the main Actor to reflect before re-attempting. This mechanism is described in contemporary work on reflective architectures and multi-agent reflection, with evidence of safety and bias mitigation under certain conditions [src_359dc8fe7c33][src_0261f15ed667][src_c59b57261d05].\n- Tool verification: a gating/verification layer that checks outputs, claims, or actions against independent evidence, retrieved context, or provenance before presenting results to users. In RAG/evidence-based generation, verification explicitly assesses fidelity to retrieved sources and context, and gate decisions on evidence support [src_0261f15ed667][src_ea237745ca52]. The mechanism emphasizes provenance, traceability, and alignment with external data sources [src_0261f15ed667, src_ea237745ca52].\n- Multi-agent specialization (MAS): an architecture where diverse agent roles (e.g., proposer, critic, verifier) and communication topologies (e.g., message passing) enable debates among specialized viewpoints. Society-of-Mind-inspired frameworks and MAD-like debate patterns structure agent personas and interactions to improve reasoning reliability, albeit with coordination overhead [src_c59b57261d05][src_ea6afdae27cf].\n\nNotes: The three patterns are not uniformly validated across domains; evidence often comes from targeted tasks, early-stage benchmarks, or controlled demonstrations. See limitations in each section and unresolved questions below [src_c59b57261d05][src_359dc8fe7c33].\n\n## Comparative Framework (6+ dimensions)\nWe compare Reflection, Tool Verification, and MAS across: Reliability (correctness, robustness, safety, consistency); Latency/Compute; Tool Integration; Observability/Traceability; Governance/Maintenance; Failure Modes; Domain Applicability; and Implementation Practicality. The table summarizes key takeaways and known tradeoffs drawn from 2023+ literature and industry reports.\n\n| Dimension | Reflection | Tool Verification | Multi-Agent Specialization (MAS) |\n|---|---|---|---|\n| Reliability (correctness, safety, bias) | Can improve safety and reduce bias in some tasks, but gains are context- and data-dependent; not universally robust without external critique or feedback loops [src_359dc8fe7c33][src_836d8b321be3]. | Enhancement of factuality when outputs are explicitly tied to verified evidence/context; depends on retrieval quality and evidence integrity [src_0261f15ed667]. | Reliability improves via diverse viewpoints and cross-checks; benefits shown on complex tasks but requires governance and can be sensitive to coordination overhead [src_c59b57261d05][src_ea6afdae27cf]. |\n| Latency/Compute | Iterative re-analysis adds additional computation; gains depend on prompt design and evaluation; some reports show marginal improvements for certain prompts [src_c59b57261d05]. | Additional verification steps introduce latency proportional to evidence retrieval and provenance checks; tool integration adds orchestration costs [src_0261f15ed667]. | Coordination across agents incurs communication and scheduling overhead; potential latency increases but can be mitigated with hierarchical roles and parallel debates [src_ea6afdae27cf][src_ea6afdae27cf]. |\n| Tool Integration / Evidence Provenance | Reflection benefits when paired with external critique but not inherently tied to tools; integration of reflective critics can be standalone or tool-augmented [src_359dc8fe7c33][src_0261f15ed667]. | Core to pattern; requires reliable sources, provenance tracking, and integration with external tools (e.g., executors, code runners) for factual grounding [src_ea237745ca52][src_0261f15ed667]. | Involves coordinating specialized modules/tools; tool coupling and provenance tracing are central to maintain safety and traceability; orchestration complexity is a key risk [src_ea6afdae27cf][src_fd0183268bfe]. |\n| Observability / Traceability | Reflective reasoning traces and diagnostics enable observability of thought processes; industry discussions highlight observability as a critical deployment requirement [src_7fcdc9ac55a4][src_036?]. | Provenance and traceability of tool outputs are essential for debugging and governance; benchmarks and traces (ToolEmu, InjecAgent, AgentDojo) illustrate robustness testing [src_ea237745ca52][src_ea237745ca52]. | Observability of inter-agent communication and decision flows is foundational; governance and auditing challenges arise from multi-agent dynamics and data handling [src_ea6afdae27cf][src_152e841d2d3d]. |\n| Governance / Maintenance | Requires careful prompt tuning and monitoring; industry coverage notes reproducibility and governance concerns; ongoing maintenance of reflective policies is nontrivial [src_7fcdc9ac55a4]. | Requires robust data provenance, tool reliability, and update practices for external components; governance of cited sources and evidence flows is critical [src_ea237745ca52]. | Governance is central due to role definitions and inter-agent protocols; maintaining lead-time for updates and ensuring alignment across roles is challenging [src_ea6afdae27cf][src_ea6afdae27cf]. |\n| Failure Modes | Context- and prompt-sensitive failures; risk of degraded performance if external critique is absent; mixed evidence across domains [src_c59b57261d05]. | Tool mis grounding, noisy retrieval, or biased evidence can introduce new failure modes; integration issues are non-trivial [src_0261f15ed667]. | Coordination failures, conflicting viewpoints, or mis-specified roles can generate cascading errors; early-stage results require cross-domain validation [src_ea6afdae27cf][src__CLAIM_c59?]. |\n| Domain Applicability | Task/data distribution dependent; some domains benefit more from reflection; others rely on external feedback or safe guardrails [src_359dc8fe7c33][src_836d8b321be3]. | High value in information-dense, citation-heavy domains (legal, medical) where provenance is crucial; depends on retrieval quality [src_0261f15ed667]. | Complex, multi-domain tasks with high reliability requirements can benefit; governance and observability overhead may be warranted in regulated contexts [src_c59b57261d05][src_ea6afdae27cf]. |\n| Practicality / Implementation | Can be implemented as modular critics or meta-prompting; evidence indicates task-dependent gains and practical caution with tuning [src_359dc8fe7c33][src_0261f15ed667]. | Requires tool integration infrastructure, provenance, and verification flows; many systems still exploring practical deployment; latency concerns exist [src_ea237745ca52]. | Requires orchestration infrastructure, inter-agent communication, and governance; early-stage but promising for hard reasoning tasks; scale and observability remain open problems [src_ea6afdae27cf][src_fd0183268bfe]. |\n\n> Note: The above table reflects current post-2023 findings, which show that none of the patterns universally outperforms the others; practical reliability often improves when patterns are combined with explicit provenance/verification and robust observability. Key sources cited reflect both empirical demonstrations and industry perspectives. See specific source notes after each claim in the text for details and limitations.\n\n## Empirical Evidence and Key Findings (2023+)\n- Reflection shows potential safety and bias mitigation benefits, but results are context- and task-dependent. Studies report safer or less biased outputs under certain prompting and feedback configurations, while other setups yield mixed or no universal gains [src_359dc8fe7c33][src_152e841d2d3d][src_836d8b321be3]. The Nature paper (2025) provides strong safety signals in controlled settings but emphasizes that gains rely on task context and evaluation design; replication across domains remains an open question [src_152e841d2d3d]. A 2024 arXiv synthesis notes variable gains depending on prompts and evaluation conditions, underscoring the need for external feedback for robust improvements [src_359dc8fe7c33]. A practical limitation is that many reflection benefits are demonstrated in early-stage or domain-specific benchmarks rather than large-scale production deployments [src_c59b57261d05].\n- MAS shows promise by enabling diverse viewpoints to detect/correct errors via debates among specialized critics; this approach is framed as scalable and training-free in targeted demonstrations, but broader benchmarking and cross-domain validation are still needed [src_c59b57261d05][src_fd0183268bfe]. The orchestration of roles such as proposer, critic, and verifier is well-documented, with Society-of-Mind-inspired and MAD-style patterns forming the basis of MAS demonstrations . However, MAS deployments face coordination latency and governance challenges that can offset some reliability gains if not carefully engineered [src_ea6afdae27cf].\n- Tool verification and provenance are central to improving factuality and reliability when tools and evidence are integrated. Verification in RAG/evidence-based generation centers on checking alignment with retrieved context and sources; gatekeeping helps reduce hallucinations, albeit with dependency on retrieval quality and context completeness [src_0261f15ed667]. Tool-use tracing and provenance frameworks (e.g., ToolEmu, InjecAgent, AgentDojo) are advancing the evaluation of robustness, but findings are still in early stages and heterogeneous across tool ecosystems [src_ea237745ca52][src_ea237745ca52]. Overall, evidence indicates that tool verification is most effective when anchored in robust provenance and well-integrated tool ecosystems .\n- Industry and governance considerations emphasize observability and reproducibility concerns when deploying reflection-based methods, highlighting the need for governance frameworks, robust observability, and controlled experimentation before production rollouts [src_7fcdc9ac55a4]. \n- In practice, several sources advocate for complementary use: reflection and verification together, with traces and provenance enabling safer and more robust decision-making. This view is supported by cross‑pattern analyses and governance discussions in Nature 2025 and related papers [src_152e841d2d3d].\n\n## Practical Guidance for Software Engineers\n- Pattern selection based on constraints:\n  - Data sensitivity and regulatory constraints: MAS with provenance-aware tool paths and traceable decision logs can support governance requirements; however, MAS adds orchestration complexity and latency that may be unsuitable for ultra-low-latency tasks without careful engineering [src_ea6afdae27cf].\n  - Latency budgets and compute constraints: Reflection adds compute for re-analysis; MIS patterns that require cross-agent debate incur extra communication and scheduling overhead; a hybrid approach that limits iterations or uses lightweight critics can help meet tight budgets [src_c59b57261d05][src_ea6afdae27cf].\n  - Tool integration maturity: If the system relies on external tools for factual grounding, ensure robust integration (standardized interfaces, retry strategies, and provenance hooks) to minimize tool-induced failures; verification should gate outputs against reliable evidence [src_0261f15ed667].\n- Hybrid design patterns for production: Consider combining reflection with verification to exploit their complementary strengths; add provenance traces to improve observability and governance. The literature frames reflection and verification as complementary, with traces enabling safer decision making [src_152e841d2d3d]. MAS can be valuable for hard reasoning tasks but requires explicit governance and observability provisions; plan for instrumentation to diagnose inter-agent failures (e.g., TRAIL to localize failures, MAST to identify multi-agent specification issues) as diagnostic aids in complex deployments [src_ea237745ca52][src_ea237745ca52].\n- Observability and testing paradigms: Invest in end-to-end observability of thought processes, tool flows, and agent communications. Industry commentary highlights reproducibility and governance as critical to successful deployment of reflection-based methods; consider adopting formal tracing and provenance standards to support debugging and audits [src_7fcdc9ac55a4]. Diagnostic frameworks like TRAIL (traces of agent actions) and MAST (multi-agent failure localization) offer structured approaches to diagnosing and localizing failures in MAS/reflection systems [src_ea237745ca52].\n- Evaluation strategy: Use 2023+ benchmarks and domain-specific evaluation to compare patterns. Early-stage work shows mixed gains for reflection; robust evaluation should include safety, bias, factuality, latency, and governance metrics; for MAS, assess inter-agent coordination latency and failure modes across tasks [src_c59b57261d05][src_ea237745ca52][src_c59b57261d05].\n\n## Limitations and Open Questions\n- Publication lag and platform heterogeneity limit generalizability: Many post-2023 results come from targeted tasks, and replication across domains remains an open problem [src_c59b57261d05].\n- Cross-pattern comparisons are sparse: Direct, replicated, head-to-head empirical comparisons across reflection, verification, and MAS are limited; most studies focus on one pattern in isolation or small-scale benchmarks [src_ea237745ca52].\n- Governance and safety frameworks for MAS remain under active development: Observability, auditing, and regulatory alignment are prominent concerns in industry discourse and early research [src_7fcdc9ac55a4][src_152e841d2d3d].\n- Generalizability across sensitive domains: Evidence of safety/bias mitigation from reflection is promising but not universal; more work is needed to understand domain-specific constraints and failure modes [src_359dc8fe7c33][src_836d8b321be3].\n\n## Conclusion\nFrom 2023 onward, reflection, tool verification, and MAS each offer distinct reliability affordances for LLM-based systems. Reflection can enhance safety and bias robustness in some contexts but is not universally effective without external critique and careful evaluation; tool verification provides a concrete path to factuality when tools and provenance are robustly integrated; MAS leverages diverse viewpoints to improve reasoning on complex tasks but introduces coordination and governance challenges. In practice, reliability appears strongest when patterns are combined with explicit provenance, observability, and governance, and when engineering tradeoffs are aligned with domain constraints (data sensitivity, latency, compliance) [src_152e841d2d3d][src_0261f15ed667][src_ea6afdae27cf]. Diagnostic frameworks (TRAIL, MAST) and structured debates offer practical aids for diagnosing and localizing failures in multi-agent/reflection systems, supporting safer deployment in production environments [src_ea237745ca52]. Engineers should tailor pattern choice to task requirements, adopt hybrid designs where appropriate, and invest in observability and governance to realize robust, maintainable LLM-based systems.\n\n## References\n\n- [src_0261f15ed667] Instruct-of-Reflection: Enhancing Large Language Models Iterative Reflection Capabilities via Dynamic-Meta Instruction — https://arxiv.org/html/2503.00902v1\n- [src_152e841d2d3d] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://www.nature.com/articles/s44387-025-00045-3\n- [src_359dc8fe7c33] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://arxiv.org/html/2406.10400v2\n- [src_7fcdc9ac55a4] Breaking Down Reflection Tuning: Enhancing LLM Performance with Self-Learning — https://arize.com/blog/breaking-down-reflection-tuning-enhancing-llm-performance-with-self-learning\n- [src_836d8b321be3] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://www.semanticscholar.org/paper/Self-Reflection-Makes-Large-Language-Models-Safer%2C-Liu-Aldahoul/47e6256ee02710c7698f49a0360718d561972e40\n- [src_c59b57261d05] Multi-Agent Reflexion Improves Reasoning Abilities in LLMs — https://arxiv.org/html/2512.20845v2\n- [src_ea237745ca52] From Agent Traces to Trust: Evidence Tracing and Execution Provenance in LLM Agents — https://arxiv.org/html/2606.04990v1\n- [src_ea6afdae27cf] The llama 3 herd of models — https://arxiv.org/abs/2407.21783\n- [src_fd0183268bfe] LLMs for Multi-Agent Cooperation — https://xue-guang.com/post/llm-marl",
    "evidence_store": {
      "sources": {
        "src_3080e417da18": {
          "id": "src_3080e417da18",
          "title": "RepEval: Effective text evaluation with LLM representation",
          "url": "https://dl.acm.org/doi/10.1145/3806262.3806263",
          "canonical_url": "https://dl.acm.org/doi/10.1145/3806262.3806263",
          "source_type": "preprint",
          "authors": [
            "Shuqian Sheng",
            "Yi Xu",
            "Tianhang Zhang",
            "Zanwei Shen",
            "Luoyi Fu",
            "Jiaxin Ding",
            "Lei Zhou",
            "Xiaoying Gan",
            "Xinbing Wang",
            "Chenghu Zhou"
          ],
          "published_date": "2024",
          "retrieved_at": "2026-08-14T01:05:59.089436+00:00"
        },
        "src_c0ab277399ff": {
          "id": "src_c0ab277399ff",
          "title": "CAMEL: Communicative Agents for “Mind” Exploration of Large Language Model Society",
          "url": "https://arxiv.org/html/2303.17760",
          "canonical_url": "https://arxiv.org/html/2303.17760",
          "source_type": "preprint",
          "authors": [
            "Li, G.",
            "et al."
          ],
          "published_date": "2023",
          "retrieved_at": "2026-08-14T01:05:59.089436+00:00"
        },
        "src_c2581ac17acf": {
          "id": "src_c2581ac17acf",
          "title": "[PDF] Towards the Next Generation of Agent Systems: From RAG to Agentic AI",
          "url": "https://www.vldb.org/2025/Workshops/VLDB-Workshops-2025/LLM+Graph/LLMGraph-8.pdf",
          "canonical_url": "https://vldb.org/2025/Workshops/VLDB-Workshops-2025/LLM+Graph/LLMGraph-8.pdf",
          "source_type": "preprint",
          "authors": [
            "Yifan Wang",
            "Haodi Ma",
            "Daisy Zhe Wang"
          ],
          "published_date": "2025",
          "retrieved_at": "2026-08-14T01:05:59.089436+00:00"
        },
        "src_c59b57261d05": {
          "id": "src_c59b57261d05",
          "title": "Multi-Agent Reflexion Improves Reasoning Abilities in LLMs",
          "url": "https://arxiv.org/html/2512.20845v2",
          "canonical_url": "https://arxiv.org/html/2512.20845v2",
          "source_type": "preprint",
          "authors": [
            "Unknown"
          ],
          "published_date": "2025",
          "retrieved_at": "2026-08-14T01:05:59.089436+00:00"
        },
        "src_ea237745ca52": {
          "id": "src_ea237745ca52",
          "title": "From Agent Traces to Trust: Evidence Tracing and Execution Provenance in LLM Agents",
          "url": "https://arxiv.org/html/2606.04990v1",
          "canonical_url": "https://arxiv.org/html/2606.04990v1",
          "source_type": "preprint",
          "authors": [
            "Ruan et al."
          ],
          "published_date": "2026",
          "retrieved_at": "2026-08-14T01:05:59.089436+00:00"
        },
        "src_fd0183268bfe": {
          "id": "src_fd0183268bfe",
          "title": "LLMs for Multi-Agent Cooperation",
          "url": "https://xue-guang.com/post/llm-marl",
          "canonical_url": "https://xue-guang.com/post/llm-marl",
          "source_type": "preprint",
          "authors": [
            "Chen, L.",
            "et al."
          ],
          "published_date": "2024",
          "retrieved_at": "2026-08-14T01:05:59.089436+00:00"
        },
        "src_0261f15ed667": {
          "id": "src_0261f15ed667",
          "title": "Instruct-of-Reflection: Enhancing Large Language Models Iterative Reflection Capabilities via Dynamic-Meta Instruction",
          "url": "https://arxiv.org/html/2503.00902v1",
          "canonical_url": "https://arxiv.org/html/2503.00902v1",
          "source_type": "preprint",
          "authors": [
            "Gao et al."
          ],
          "published_date": "2025-03",
          "retrieved_at": "2026-08-14T01:06:54.966531+00:00"
        },
        "src_152e841d2d3d": {
          "id": "src_152e841d2d3d",
          "title": "Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral",
          "url": "https://www.nature.com/articles/s44387-025-00045-3",
          "canonical_url": "https://nature.com/articles/s44387-025-00045-3",
          "source_type": "peer_reviewed",
          "authors": [
            "Zhibin Gou",
            "Zhihong Shao",
            "Weizhu Chen"
          ],
          "published_date": "2025",
          "retrieved_at": "2026-08-14T01:06:54.966531+00:00"
        },
        "src_7fcdc9ac55a4": {
          "id": "src_7fcdc9ac55a4",
          "title": "Breaking Down Reflection Tuning: Enhancing LLM Performance with Self-Learning",
          "url": "https://arize.com/blog/breaking-down-reflection-tuning-enhancing-llm-performance-with-self-learning",
          "canonical_url": "https://arize.com/blog/breaking-down-reflection-tuning-enhancing-llm-performance-with-self-learning",
          "source_type": "industry",
          "authors": [
            "Sarah Welsh"
          ],
          "published_date": "2024-09-19",
          "retrieved_at": "2026-08-14T01:06:54.966531+00:00"
        },
        "src_836d8b321be3": {
          "id": "src_836d8b321be3",
          "title": "Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral",
          "url": "https://www.semanticscholar.org/paper/Self-Reflection-Makes-Large-Language-Models-Safer%2C-Liu-Aldahoul/47e6256ee02710c7698f49a0360718d561972e40",
          "canonical_url": "https://semanticscholar.org/paper/Self-Reflection-Makes-Large-Language-Models-Safer%2C-Liu-Aldahoul/47e6256ee02710c7698f49a0360718d561972e40",
          "source_type": "peer_reviewed",
          "authors": [
            "Unknown"
          ],
          "published_date": "2024",
          "retrieved_at": "2026-08-14T01:06:54.966531+00:00"
        },
        "src_c2d5353f6595": {
          "id": "src_c2d5353f6595",
          "title": "ReflectEvo: Improving Meta Introspection of Small LLMs by ...",
          "url": "https://aclanthology.org/2025.findings-acl.871.pdf",
          "canonical_url": "https://aclanthology.org/2025.findings-acl.871.pdf",
          "source_type": "peer_reviewed",
          "authors": [
            "Ming Li",
            "Lichang Chen",
            "Jiuhai Chen",
            "Shwai He",
            "Heng Huang",
            "Jiuxiang Gu",
            "Tianyi Zhou"
          ],
          "published_date": "2025",
          "retrieved_at": "2026-08-14T01:06:54.966531+00:00"
        },
        "src_ea6afdae27cf": {
          "id": "src_ea6afdae27cf",
          "title": "The llama 3 herd of models",
          "url": "https://arxiv.org/abs/2407.21783",
          "canonical_url": "https://arxiv.org/abs/2407.21783",
          "source_type": "preprint",
          "authors": [
            "Dubey et al."
          ],
          "published_date": "2024",
          "retrieved_at": "2026-08-14T01:06:54.966531+00:00"
        },
        "src_359dc8fe7c33": {
          "id": "src_359dc8fe7c33",
          "title": "Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral",
          "url": "https://arxiv.org/html/2406.10400v2",
          "canonical_url": "https://arxiv.org/html/2406.10400v2",
          "source_type": "preprint",
          "authors": [],
          "published_date": "2024-06",
          "retrieved_at": "2026-08-14T01:08:24.088447+00:00"
        },
        "src_971edd2cf82c": {
          "id": "src_971edd2cf82c",
          "title": "GitHub - rxlqn/awesome-llm-self-reflection: augmented LLM with self reflection",
          "url": "https://github.com/rxlqn/awesome-llm-self-reflection",
          "canonical_url": "https://github.com/rxlqn/awesome-llm-self-reflection",
          "source_type": "other",
          "authors": [],
          "published_date": null,
          "retrieved_at": "2026-08-14T01:08:24.088447+00:00"
        },
        "src_f940fa9ff9b8": {
          "id": "src_f940fa9ff9b8",
          "title": "Self-Reflection in LLM Agents: Effects on Problem-Solving Performance",
          "url": "https://arxiv.org/pdf/2405.06682",
          "canonical_url": "https://arxiv.org/abs/2405.06682",
          "source_type": "preprint",
          "authors": [
            "Renze, M.",
            "Guven, E."
          ],
          "published_date": "2024-05",
          "retrieved_at": "2026-08-14T01:08:24.088447+00:00"
        }
      },
      "claims": {
        "clm_60aaa5f12151": {
          "id": "clm_60aaa5f12151",
          "claim": "Reflection in LLM systems is an iterative self-analysis loop where the agent re-examines prior reasoning and intermediate results to guide subsequent attempts.",
          "source_ids": [
            "src_c59b57261d05"
          ],
          "evidence_excerpt": "A multi-agent reflective framework is described in which diverse critics analyze failed reasoning and propose corrective strategies, enabling the Actor to reflect before re-attempting.",
          "confidence": "high",
          "limitations": [
            "Evidence drawn from a single or limited set of preprints; generalizability to all domains and production-scale systems is unproven."
          ]
        },
        "clm_28d4d989f630": {
          "id": "clm_28d4d989f630",
          "claim": "Multi-agent specialization (MAS) improves reliability by leveraging diverse agent perspectives to detect and correct errors during reasoning.",
          "source_ids": [
            "src_c59b57261d05",
            "src_fd0183268bfe"
          ],
          "evidence_excerpt": "The literature describes MAS as offering a scalable, training-free path toward more reliable and self-correcting LLM agents via debates among specialized critics.",
          "confidence": "high",
          "limitations": [
            "Predominantly demonstrated in targeted tasks; broader benchmarks and production-scale validations are needed."
          ]
        },
        "clm_5a2882f8f8ff": {
          "id": "clm_5a2882f8f8ff",
          "claim": "MAS architectures commonly employ roles (e.g., proposer, critic, verifier) and communication topologies (e.g., message passing) to coordinate diverse viewpoints.",
          "source_ids": [
            "src_c59b57261d05"
          ],
          "evidence_excerpt": "The Society-of-Mind-inspired framework and MAD-like debate paradigms are used to structure agent personas and interactions.",
          "confidence": "high",
          "limitations": [
            "Prominent in conceptual demonstrations; concrete, scalable deployment patterns require more empirical study."
          ]
        },
        "clm_c57be95b4367": {
          "id": "clm_c57be95b4367",
          "claim": "Tool verification adds a gating mechanism that checks tool outputs and claims against independent evidence or retrieved context before releasing results.",
          "source_ids": [
            "src_ea237745ca52"
          ],
          "evidence_excerpt": "Verification checks whether claims, actions, or outputs are supported by appropriate evidence, integrating with evidence and provenance considerations in tool use.",
          "confidence": "high",
          "limitations": [
            "Empirical validation across diverse tool ecosystems and real-world deployments remains limited."
          ]
        },
        "clm_dad55f57baed": {
          "id": "clm_dad55f57baed",
          "claim": "Tool-use tracing and provenance are central to modern verification efforts, with defenses and benchmarks evaluating how information and tool outputs flow through agent execution.",
          "source_ids": [
            "src_ea237745ca52"
          ],
          "evidence_excerpt": "Recent work emphasizes tracing tool use and execution provenance, using benchmarks like ToolEmu, InjecAgent, and AgentDojo to evaluate robustness and instruction-following.",
          "confidence": "high",
          "limitations": [
            "Many findings are early-stage, with heterogeneous tool ecosystems and limited cross-domain validation."
          ]
        },
        "clm_f7c4e8e923ea": {
          "id": "clm_f7c4e8e923ea",
          "claim": "Verification in RAG-enabled and evidence-based generation emphasizes fidelity to retrieved context and cited sources as a core correctness constraint.",
          "source_ids": [
            "src_ea237745ca52"
          ],
          "evidence_excerpt": "In RAG and evidence-based generation, verification asks whether generated claims are faithful to retrieved context or cited sources.",
          "confidence": "high",
          "limitations": [
            "Dependence on retrieval quality and context completeness; practical systems may struggle with noisy or partial data."
          ]
        },
        "clm_3059cb655861": {
          "id": "clm_3059cb655861",
          "claim": "There exist growth areas and evaluation frameworks (e.g., TRAIL, MAST) for diagnosing and localizing failures in multi-agent/reflective systems.",
          "source_ids": [
            "src_ea237745ca52"
          ],
          "evidence_excerpt": "TRAIL uses annotated agent traces to localize the step, component, or interaction responsible for a failure; MAST highlights that multi-agent failures can arise from specification.",
          "confidence": "medium",
          "limitations": [
            "Trails and diagnostic frameworks are still maturing and not yet universally adopted across platforms."
          ]
        },
        "clm_bf58b088ea27": {
          "id": "clm_bf58b088ea27",
          "claim": "Evidence from 2023+ sources indicates that reflection, when complemented with verification and traceability, can improve reliability and safety in LLM-based systems rather than operate in a vacuum.",
          "source_ids": [
            "src_c59b57261d05",
            "src_ea237745ca52"
          ],
          "evidence_excerpt": "The literature frames reflection and verification as complementary patterns, with traces and provenance enabling safer and more robust decision making.",
          "confidence": "medium",
          "limitations": [
            "Direct experimental comparisons across all three patterns are sparse in the available sources."
          ]
        },
        "clm_03d249787afa": {
          "id": "clm_03d249787afa",
          "claim": "Reflection yields context-dependent reliability gains with potential safety benefits, but not universal improvements without external feedback.",
          "source_ids": [
            "src_0261f15ed667",
            "src_152e841d2d3d",
            "src_836d8b321be3"
          ],
          "evidence_excerpt": "Nature 2025 reports safety and bias mitigation benefits from self-reflection in certain contexts, while arXiv 2025 work notes mixed outcomes and potential degradation without external critique; a 2024 synthesis notes variable gains depending on prompts and evaluation conditions.",
          "confidence": "medium",
          "limitations": [
            "Context-dependent results; limited cross-domain generalization; some studies report no consistent improvements without external input."
          ]
        },
        "clm_28b0a22936d0": {
          "id": "clm_28b0a22936d0",
          "claim": "Tool verification can improve factuality when tools are properly integrated and reliable, but benefits depend on tool availability, verification, and integration effort.",
          "source_ids": [
            "src_0261f15ed667"
          ],
          "evidence_excerpt": "The Instruct-of-Reflection work describes a CRITIC framework that uses external tools (such as code executors) to obtain feedback and reflect on outputs.",
          "confidence": "medium",
          "limitations": [
            "Tool reliability and integration complexity; potential latency; external tool failures can become new error sources."
          ]
        },
        "clm_ce3a177eda39": {
          "id": "clm_ce3a177eda39",
          "claim": "Multi-agent specialization can improve reliability on complex tasks via coordinated roles, but incurs coordination latency and governance challenges.",
          "source_ids": [
            "src_ea6afdae27cf"
          ],
          "evidence_excerpt": "The llama 3 herd paper demonstrates benefits from a coordinated set of specialized models; however, orchestration complexity and inter-agent communication can impact latency and observability.",
          "confidence": "medium",
          "limitations": [
            "Coordination overhead; early-stage results; not yet broadly validated across domains."
          ]
        },
        "clm_9f59eacd37e6": {
          "id": "clm_9f59eacd37e6",
          "claim": "Reflection-based safety improvements have been observed, but the gains are not guaranteed and depend on task, data, and evaluation context.",
          "source_ids": [
            "src_152e841d2d3d",
            "src_836d8b321be3"
          ],
          "evidence_excerpt": "Nature 2025 reports safer/less biased LLMs with self-reflection under certain conditions; semantics summary cites similar outcomes in 2024; context dependence noted.",
          "confidence": "medium",
          "limitations": [
            "Not universal; dependent on evaluation metrics and prompts; external feedback may be necessary."
          ]
        },
        "clm_36f62dbf70a1": {
          "id": "clm_36f62dbf70a1",
          "claim": "Industry coverage highlights reproducibility concerns and engineering challenges in applying reflection-based methods, emphasizing observability and governance needs.",
          "source_ids": [
            "src_7fcdc9ac55a4"
          ],
          "evidence_excerpt": "Engineering blogs discuss caution about reproducibility, drama around reflection tuning, and the need for robust observability and governance when deploying reflection-based approaches.",
          "confidence": "medium",
          "limitations": [
            "Industry/blog perspective; limited formal evaluation; generalizability uncertain."
          ]
        },
        "clm_bbae6721fb74": {
          "id": "clm_bbae6721fb74",
          "claim": "Domain- and constraint-aware pattern selection is essential; data sensitivity and latency budgets influence whether reflection, tool use, or multi-agent approaches are preferred.",
          "source_ids": [
            "src_152e841d2d3d"
          ],
          "evidence_excerpt": "Nature 2025 discusses governance and data-handling concerns that influence pattern selection; tool use and multi-agent patterns require observability and compliance considerations.",
          "confidence": "medium",
          "limitations": [
            "Broad guidance; needs organization-specific evaluation."
          ]
        },
        "clm_a78bb05a29fd": {
          "id": "clm_a78bb05a29fd",
          "claim": "Self-reported and industry narratives indicate that combining reflection with tool use and modular agents may be advantageous, but rigorous, standardized benchmarks are needed to guide integration decisions.",
          "source_ids": [
            "src_0261f15ed667",
            "src_7fcdc9ac55a4",
            "src_c2d5353f6595"
          ],
          "evidence_excerpt": "Preliminary results suggest potential gains from combining patterns; guidance remains contingent on evaluation design and tooling maturity.",
          "confidence": "low",
          "limitations": [
            "Limited cross-study comparability; early-stage integration frameworks; more benchmarks needed."
          ]
        },
        "clm_1a596262ef60": {
          "id": "clm_1a596262ef60",
          "claim": "Self-reflection can improve safety and reduce bias in LLM outputs.",
          "source_ids": [
            "src_359dc8fe7c33"
          ],
          "evidence_excerpt": "Self-reflection approaches have been reported to make LLM outputs safer and less biased, though results are not universally consistent across tasks.",
          "confidence": "high",
          "limitations": [
            "Evidence is drawn from a subset of experiments with variable results; generalizability across domains remains uncertain."
          ]
        },
        "clm_57f49e39276d": {
          "id": "clm_57f49e39276d",
          "claim": "The benefits of self-reflection are task- and data-dependent, with some setups showing limited or mixed gains.",
          "source_ids": [
            "src_359dc8fe7c33"
          ],
          "evidence_excerpt": "Reports note conflicting results and limited improvements in some contexts, indicating gains depend on task and data distribution.",
          "confidence": "high",
          "limitations": [
            "Most evidence from early-stage studies; broader replication is needed."
          ]
        },
        "clm_699bb1b2ea33": {
          "id": "clm_699bb1b2ea33",
          "claim": "Self-reflection can be augmented by external feedback (humans or other models) to guide corrections.",
          "source_ids": [
            "src_359dc8fe7c33"
          ],
          "evidence_excerpt": "External feedback signals have been proposed to steer self-reflection and improve safety and quality.",
          "confidence": "medium",
          "limitations": [
            "Specific mechanisms for integrating external feedback vary; not universally established."
          ]
        },
        "clm_7e059e996b97": {
          "id": "clm_7e059e996b97",
          "claim": "ReflectEvo investigates meta introspection in small LLMs to improve reasoning capabilities.",
          "source_ids": [
            "src_c2d5353f6595"
          ],
          "evidence_excerpt": "ReflectEvo proposes improvements to meta introspection in small LLMs, demonstrating enhanced rationality under certain conditions.",
          "confidence": "medium",
          "limitations": [
            "Limited to small LLMs; generalization to larger models not guaranteed."
          ]
        },
        "clm_a927664af511": {
          "id": "clm_a927664af511",
          "claim": "Self-reflection in LLM agents has demonstrated effects on problem-solving performance.",
          "source_ids": [
            "src_f940fa9ff9b8"
          ],
          "evidence_excerpt": "Studies report that self-reflection can influence problem-solving performance in LLM agents.",
          "confidence": "medium",
          "limitations": [
            "Results depend on task type and reflection prompting; more broad empirical work is needed."
          ]
        },
        "clm_ddc5816e1a0c": {
          "id": "clm_ddc5816e1a0c",
          "claim": "Self-reflection-enhanced LLMs have been reported to be safer and more ideologically neutral in some analyses.",
          "source_ids": [
            "src_152e841d2d3d"
          ],
          "evidence_excerpt": "Nature-published analyses suggest self-reflection can improve safety and reduce ideological bias in LLMs.",
          "confidence": "medium",
          "limitations": [
            "Nature article provides broad evidence; domain-specific details may vary."
          ]
        },
        "clm_42fbaacfc3ec": {
          "id": "clm_42fbaacfc3ec",
          "claim": "Engineering practice on self-reflection is documented in open-source resources (e.g., ‘awesome-llm-self-reflection’).",
          "source_ids": [
            "src_971edd2cf82c"
          ],
          "evidence_excerpt": "GitHub repositories compile and discuss practical self-reflection techniques and case studies for LLMs.",
          "confidence": "low",
          "limitations": [
            "Repository content is heterogeneous and not peer‑reviewed; evidence strength varies."
          ]
        },
        "clm_1ec9f726ba6a": {
          "id": "clm_1ec9f726ba6a",
          "claim": "Meta-introspection and related self-reflection strategies are discussed in the 2025 ACL Findings paper (ReflectEvo) with references to verbal reinforcement learning and related approaches.",
          "source_ids": [
            "src_c2d5353f6595"
          ],
          "evidence_excerpt": "The Findings paper references frameworks like verbal reinforcement learning and self-reflection strategies as part of introspective agent design.",
          "confidence": "low",
          "limitations": [
            "References are to a survey within a Findings paper; specific empirical validations of these sub-strategies are not the focus."
          ]
        }
      },
      "packets": [
        {
          "worker_id": 1,
          "task": "From 2023 onward, what are the formal definitions, mechanisms, and observed failure modes of reflection, tool verification, and multi-agent specialization in LLM systems, and how do researchers compare their functional behaviors?",
          "summary": "This research packet synthesizes post-2023 literature on three reliability patterns for LLM-based systems—reflection, tool verification, and multi-agent specialization (MAS)—to compare definitions, mechanisms, observed failure modes, and practical implications for engineering deployment. The assembled sources indicate that reflection involves iterative self-analysis and debate among agents to revise prior reasoning; MAS leverages diverse agent roles and communication topologies to improve robustness through distributed judgment; verification introduces external checks, traceability, and provenance to constrain tool use and ensure claims are aligned with evidence. Across patterns, empirical work (often with preprints) reports improved problem-solving reliability and safety via diversification of viewpoints (MAS) or through gated outputs and evidence-based checks (verification). However, most results are early-stage, task-specific, and frequently rely on arXiv preprints or conference/workshop materials, with limited large-scale production validation. Several sources highlight the need for traceability (execution provenance and tool-use traces), evaluation frameworks (TRAIL, MAST), and integration considerations (latency, compute, observability). A cross-pattern view suggests potential synergy—reflection can amplify correct reasoning when paired with verification gates and traceable tool use—but also cautions about potential bias amplification or coordination overhead. Engineers should anticipate platform heterogeneity, evolving tooling, and the current lack of standardized benchmarks when selecting and combining these patterns.",
          "source_ids": [
            "src_3080e417da18",
            "src_c0ab277399ff",
            "src_c2581ac17acf",
            "src_c59b57261d05",
            "src_ea237745ca52",
            "src_fd0183268bfe"
          ],
          "claim_ids": [
            "clm_60aaa5f12151",
            "clm_28d4d989f630",
            "clm_5a2882f8f8ff",
            "clm_c57be95b4367",
            "clm_dad55f57baed",
            "clm_f7c4e8e923ea",
            "clm_3059cb655861",
            "clm_bf58b088ea27"
          ],
          "unresolved_questions": [
            "What standardized benchmarks exist (or should be developed) to compare reflection, tool verification, and MAS across tasks and domains in 2023+ literature?",
            "How do latency budgets, compute constraints, and memory limits influence the practical deployment of each pattern (and their combinations) in production systems?",
            "What principled guidelines exist for combining these patterns (e.g., reflection+verification+MAS) to maximize reliability without incurring prohibitive overhead?",
            "How generalizable are current 2023+ findings across domains with different data sensitivities, regulatory requirements, and threat models?",
            "What governance, auditing, and safety frameworks are appropriate for MAS and tool-use verification in regulated industries?",
            "Which patterns offer the best trade-offs for specific domains (e.g., finance, healthcare, safety-critical systems) given typical latency and reliability requirements?"
          ]
        },
        {
          "worker_id": 2,
          "task": "Across 2023 and later, what empirical evidence, evaluation methodologies, and benchmarks exist comparing reflection, tool verification, and multi-agent specialization across key dimensions (reliability, safety, robustness, latency/compute, tool integration, observability, governance/maintenance, and domain applicability)?",
          "summary": "This research packet synthesizes 2023+ empirical and engineering evidence on three reliability patterns for LLM agents—reflection, tool verification, and multi-agent specialization—and maps them to concrete evaluation methodologies, benchmarks, and deployment considerations. Evidence indicates that reflection can yield context-dependent safety and problem-solving benefits but often produces mixed gains that depend on prompts, evaluation contexts, and the presence of external feedback. Tool verification tends to improve factuality and currency when tool access is reliable and well-integrated, at the expense of latency and system complexity. Multi-agent specialization shows promise for tackling complex, domain-diverse tasks by distributing roles across specialized sub-agents, but introduces orchestration, observability, and governance overhead. Across these patterns, empirical work from 2023 onward emphasizes task- and environment-specific performance, the importance of observability and provenance, and the need for careful pattern-combination strategies under latency and regulatory constraints. The evidence base includes peer-reviewed conferences and industry engineering analyses, with notable sources from 2025 (reflective safety, meta-introspection) and 2024–2025 engineering discussions on reflection tuning and tool use. Practically, engineers should consider pattern selection or combination based on latency budgets, data governance, and measurable reliability targets, while deploying rigorous evaluation, observability, and governance practices.",
          "source_ids": [
            "src_0261f15ed667",
            "src_152e841d2d3d",
            "src_7fcdc9ac55a4",
            "src_836d8b321be3",
            "src_c2d5353f6595",
            "src_ea6afdae27cf"
          ],
          "claim_ids": [
            "clm_03d249787afa",
            "clm_28b0a22936d0",
            "clm_ce3a177eda39",
            "clm_9f59eacd37e6",
            "clm_36f62dbf70a1",
            "clm_bbae6721fb74",
            "clm_a78bb05a29fd"
          ],
          "unresolved_questions": [
            "What standardized metrics and benchmarks can fairly compare reflection, tool verification, and multi-agent specialization across diverse deployments (different LLM providers, tool stacks, and tasks)?",
            "How can we separate tool-integration failures from core model reliability in evaluations of tool verification patterns?",
            "What are best-practice design patterns for safely combining reflection, tool use, and multi-agent orchestration without pushing latency beyond acceptable budgets?",
            "What governance and observability requirements are needed for scalable deployment (decision provenance, tool provenance, agent role definitions, audits)?",
            "Which regulatory constraints (data handling, privacy, licensing) most strongly constrain the use of each pattern in sensitive domains?",
            "How do model scale, task type, and domain affect the relative effectiveness of reflection, tool verification, and multi-agent approaches, and can we map these effects into a decision framework for engineers?"
          ]
        },
        {
          "worker_id": 3,
          "task": "What practical engineering guidance emerges for software engineers implementing these patterns in LLM-based systems, including when to use which pattern or combination, actionable design recommendations, deployment constraints (latency, compute, memory), observability and testing strategies, governance considerations, data sensitivity issues, and open questions for future work?",
          "summary": "Across the 2023–present literature, reflection, tool verification, and multi-agent specialization each offer distinct reliability benefits and engineering trade-offs for LLM-based systems. Reflection provides internal quality control by prompting the model to critique or revise its own reasoning, offering potential safety and bias reductions but with results that are task- and data-dependent and prone to new biases if the critique logic is weak. Tool verification improves factuality and currency by grounding outputs in external tools, at the cost of additional latency, integration complexity, and data governance considerations. Multi-agent specialization distributes tasks across specialized roles to improve robustness and problem-solving for complex tasks, but introduces orchestration, traceability, and coordination challenges. Given these trade-offs, production-grade systems are likely to benefit from hybrid architectures that combine these patterns in a principled way (e.g., reflection with tool verification, or tool-enabled workflows with a validation layer), while prioritizing observability, latency budgeting, and data governance. The evidence base includes self-reflection-focused studies, meta-introspection investigations, agent-based reflection work, and engineering resources, including peer-reviewed and open-source sources from 2023–present.",
          "source_ids": [
            "src_152e841d2d3d",
            "src_359dc8fe7c33",
            "src_971edd2cf82c",
            "src_c2d5353f6595",
            "src_f940fa9ff9b8"
          ],
          "claim_ids": [
            "clm_1a596262ef60",
            "clm_57f49e39276d",
            "clm_699bb1b2ea33",
            "clm_7e059e996b97",
            "clm_a927664af511",
            "clm_ddc5816e1a0c",
            "clm_42fbaacfc3ec",
            "clm_1ec9f726ba6a"
          ],
          "unresolved_questions": [
            "How can we quantify and compare reliability gains across reflection, tool verification, and multi-agent specialization in production environments?",
            "What principled guidelines exist for combining the three patterns without incurring prohibitive latency or governance complexity?",
            "What standardized observability and auditing frameworks can span single-model, tool-based, and multi-agent deployments?",
            "How should introspection traces and tool-output provenance be governed to protect data privacy and regulatory compliance?",
            "How portable are these patterns across different LLM providers and tool ecosystems?",
            "What benchmarks and evaluation suites best capture safety, reliability, and governance across 2023+ evidence?"
          ]
        }
      ],
      "unresolved_questions": [
        "What standardized benchmarks exist (or should be developed) to compare reflection, tool verification, and MAS across tasks and domains in 2023+ literature?",
        "How do latency budgets, compute constraints, and memory limits influence the practical deployment of each pattern (and their combinations) in production systems?",
        "What principled guidelines exist for combining these patterns (e.g., reflection+verification+MAS) to maximize reliability without incurring prohibitive overhead?",
        "How generalizable are current 2023+ findings across domains with different data sensitivities, regulatory requirements, and threat models?",
        "What governance, auditing, and safety frameworks are appropriate for MAS and tool-use verification in regulated industries?",
        "Which patterns offer the best trade-offs for specific domains (e.g., finance, healthcare, safety-critical systems) given typical latency and reliability requirements?",
        "What standardized metrics and benchmarks can fairly compare reflection, tool verification, and multi-agent specialization across diverse deployments (different LLM providers, tool stacks, and tasks)?",
        "How can we separate tool-integration failures from core model reliability in evaluations of tool verification patterns?",
        "What are best-practice design patterns for safely combining reflection, tool use, and multi-agent orchestration without pushing latency beyond acceptable budgets?",
        "What governance and observability requirements are needed for scalable deployment (decision provenance, tool provenance, agent role definitions, audits)?",
        "Which regulatory constraints (data handling, privacy, licensing) most strongly constrain the use of each pattern in sensitive domains?",
        "How do model scale, task type, and domain affect the relative effectiveness of reflection, tool verification, and multi-agent approaches, and can we map these effects into a decision framework for engineers?",
        "How can we quantify and compare reliability gains across reflection, tool verification, and multi-agent specialization in production environments?",
        "What principled guidelines exist for combining the three patterns without incurring prohibitive latency or governance complexity?",
        "What standardized observability and auditing frameworks can span single-model, tool-based, and multi-agent deployments?",
        "How should introspection traces and tool-output provenance be governed to protect data privacy and regulatory compliance?",
        "How portable are these patterns across different LLM providers and tool ecosystems?",
        "What benchmarks and evaluation suites best capture safety, reliability, and governance across 2023+ evidence?"
      ]
    },
    "error": null
  },
  "grades": [
    {
      "grader": "scoping_behavior",
      "passed": true,
      "score": 1.0,
      "issues": [],
      "details": {
        "clarification_requested": 0,
        "clarification_resolved": 0
      }
    },
    {
      "grader": "outcome_contract",
      "passed": false,
      "score": 0.5,
      "issues": [
        "Word count 2044 is outside [1200, 1800].",
        "Missing sections: ['Limitations']"
      ],
      "details": {
        "word_count": 2044,
        "source_count": 15
      }
    },
    {
      "grader": "trajectory",
      "passed": true,
      "score": 1.0,
      "issues": [],
      "details": {
        "tool_calls": 12,
        "worker_failures": 0,
        "guard_rejections": 0
      }
    },
    {
      "grader": "evidence_provenance",
      "passed": false,
      "score": 0.8571428571428572,
      "issues": [
        "Malformed source citation tokens: ['src_036?', 'src__CLAIM_c59?']"
      ],
      "details": {
        "cited_source_ids": 9,
        "stored_sources": 15,
        "stored_claims": 23,
        "body_source_ids": 9,
        "reference_source_ids": 9,
        "malformed_source_tokens": [
          "src_036?",
          "src__CLAIM_c59?"
        ],
        "applicable": true
      }
    },
    {
      "grader": "semantic_report",
      "passed": false,
      "score": 0.7875,
      "issues": [
        "Placeholders and missing source mappings in Comparative Framework table (e.g., [src_036?], [src__CLAIM_c59?]) are inconsistently cited.",
        "Many claims rely on early-stage results from preprints or non-peer‑reviewed sources; evidence base is task‑specific and not yet broadly validated.",
        "Several sources are industry blogs or open‑source repositories rather than peer‑reviewed studies, which weakens formal evidentiary strength.",
        "Some citations (e.g., Nature 2025) are referenced alongside less corroborated arXiv work, risking overgeneralization across domains.",
        "Report contains mixed confidence notes for certain claims (e.g., ReflectEvo) that are not consistently aligned with the cited sources.",
        "There is an acknowledged lack of standardized benchmarks, yet the report does not specify a clear, reproducible benchmarking framework across patterns to rigorously compare them."
      ],
      "details": {
        "scope_fidelity": 0.85,
        "evidence_fidelity": 0.72,
        "synthesis_quality": 0.8,
        "limitation_quality": 0.78,
        "issues": [
          "Placeholders and missing source mappings in Comparative Framework table (e.g., [src_036?], [src__CLAIM_c59?]) are inconsistently cited.",
          "Many claims rely on early-stage results from preprints or non-peer‑reviewed sources; evidence base is task‑specific and not yet broadly validated.",
          "Several sources are industry blogs or open‑source repositories rather than peer‑reviewed studies, which weakens formal evidentiary strength.",
          "Some citations (e.g., Nature 2025) are referenced alongside less corroborated arXiv work, risking overgeneralization across domains.",
          "Report contains mixed confidence notes for certain claims (e.g., ReflectEvo) that are not consistently aligned with the cited sources.",
          "There is an acknowledged lack of standardized benchmarks, yet the report does not specify a clear, reproducible benchmarking framework across patterns to rigorously compare them."
        ],
        "explanation": "The report largely covers the requested patterns and offers a coherent synthesis, but evidence mapping is imperfect (some placeholders, reliance on early-stage/preprint sources, and non-peer-reviewed materials). There is moderate alignment with the requested scope and a reasonable synthesis, yet methodological gaps (benchmarks, cross-domain validation) limit evidentiary strength and reproducibility."
      }
    }
  ],
  "overall_score": 0.8289285714285715,
  "passed": false
}
''')
print("Embedded V3 baseline fixture loaded.")


In [20]:
REPLAY_SAVED_EVAL = False


def regrade_saved_result(
    case: EvalCase, previous: EvalRunResult
) -> EvalRunResult:
    grades = [
        grade_scoping_behavior(case, previous.trace),
        grade_outcome_contract(case, previous.trace),
        grade_trajectory(case, previous.trace),
    ]
    if previous.trace.evidence_store is not None:
        grades.append(grade_evidence_provenance(previous.trace))
    semantic_grade = next(
        (grade for grade in previous.grades if grade.grader == "semantic_report"),
        None,
    )
    if semantic_grade is not None:
        grades.append(semantic_grade)
    overall_score = statistics.fmean(grade.score for grade in grades)
    return EvalRunResult(
        case_id=previous.case_id,
        workflow_version=previous.workflow_version,
        trace=previous.trace,
        grades=grades,
        overall_score=overall_score,
        passed=all(grade.passed for grade in grades),
    )


if REPLAY_SAVED_EVAL:
    saved_payload = [V3_BASELINE_RESULT_PAYLOAD]
    saved_results = [EvalRunResult.model_validate(item) for item in saved_payload]
    cases_by_id = {case.id: case for case in EVAL_CASES}
    regraded_results = [
        regrade_saved_result(cases_by_id[result.case_id], result)
        for result in saved_results
    ]
    regraded_comparison = render_comparison_markdown(
        aggregate_eval_results(regraded_results)
    )
    regraded_details = render_eval_details(regraded_results)
    show("Regraded Saved V4 Evaluation", regraded_details)

# 10. Evaluation-Driven V3.1 Hardening

V3 remains the independent evidence-aware baseline. V4 owns the improved candidate because
the changes below are derived from observed evaluation failures, not from the original V3
design assumptions.

The controlled experiment reuses the saved V3 trace and `EvidenceStore`. It replaces only
the final report stage, so a score change can be attributed to the report contract and
citation harness rather than to a different web search or a different evidence sample.

```text
saved V3 trace + same EvidenceStore
                |
                v
bounded report revision -> deterministic citation canonicalization
                |
                v
contract guard -> same V4 graders -> before/after comparison
```

This is a **replay evaluation**, not a new end-to-end research run. A later release can run
V3.1 from scoping through research after the isolated change passes this controlled test.


## 10.1 Report Contract and Fail-Closed Citation Guard

Prompt instructions improve the probability of a valid report; they do not enforce the
contract. Python therefore checks the observable requirements after every revision:

- exact required headings and requested concepts;
- total word-count bounds and minimum cited-source count;
- exact stable source IDs only;
- no placeholder-like `src_*` tokens;
- one canonical References entry, with the stored URL, for every body citation.

A failed check becomes feedback for one bounded revision round. If the final attempt still
fails, the harness raises an error instead of publishing a plausible-looking invalid report.


In [21]:
V31_EXACT_SOURCE_ID_PATTERN = re.compile(r"src_[0-9a-f]{12}")
V31_SOURCE_LIKE_TOKEN_PATTERN = re.compile(r"\bsrc_[A-Za-z0-9_?]+")
V31_REFERENCES_HEADING_PATTERN = re.compile(
    r"^## References\s*$", re.MULTILINE | re.IGNORECASE
)


class V31ReportRevision(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    markdown: str = Field(min_length=1)
    used_source_ids: list[str] = Field(default_factory=list)
    used_claim_ids: list[str] = Field(default_factory=list)


class V31ContractCheck(BaseModel):
    model_config = ConfigDict(extra="forbid")

    passed: bool
    issues: list[str]
    word_count: int = Field(ge=0)
    cited_source_ids: list[str]
    malformed_source_tokens: list[str]
    unknown_source_ids: list[str]
    missing_sections: list[str]


def split_v31_report(markdown: str) -> tuple[str, str]:
    heading = V31_REFERENCES_HEADING_PATTERN.search(markdown)
    if heading is None:
        return markdown.rstrip(), ""
    return markdown[: heading.start()].rstrip(), markdown[heading.end() :].strip()


def find_malformed_source_tokens(markdown: str) -> list[str]:
    exact_ids = set(V31_EXACT_SOURCE_ID_PATTERN.findall(markdown))
    source_like_tokens = set(V31_SOURCE_LIKE_TOKEN_PATTERN.findall(markdown))
    return sorted(source_like_tokens - exact_ids)


def build_v31_source_aliases(
    store: EvidenceStore,
) -> tuple[dict[str, str], dict[str, str]]:
    source_to_alias = {
        source_id: f"S{index:02d}"
        for index, source_id in enumerate(sorted(store.sources), start=1)
    }
    alias_to_source = {
        alias: source_id for source_id, alias in source_to_alias.items()
    }
    return source_to_alias, alias_to_source


def build_v31_evidence_catalog(store: EvidenceStore) -> dict[str, Any]:
    source_to_alias, _ = build_v31_source_aliases(store)
    sources = [
        {
            "alias": source_to_alias[source.id],
            "title": source.title,
            "url": source.url,
            "source_type": source.source_type,
            "authors": source.authors,
            "published_date": source.published_date,
        }
        for source in store.sources.values()
    ]
    claims = [
        {
            "claim": claim.claim,
            "source_aliases": [
                source_to_alias[source_id]
                for source_id in claim.source_ids
                if source_id in source_to_alias
            ],
            "evidence_excerpt": claim.evidence_excerpt,
            "confidence": claim.confidence,
            "limitations": claim.limitations,
        }
        for claim in store.claims.values()
    ]
    return {"sources": sources, "claims": claims}


def prepare_v31_alias_report(markdown: str, store: EvidenceStore) -> str:
    source_to_alias, _ = build_v31_source_aliases(store)
    result = markdown
    for source_id, alias in source_to_alias.items():
        result = result.replace(source_id, alias)
    result = V31_SOURCE_LIKE_TOKEN_PATTERN.sub("", result)
    result = re.sub(r"\[\s*(?:,\s*)*\]", "", result)
    return result


def materialize_v31_source_aliases(
    markdown: str, store: EvidenceStore
) -> str:
    _, alias_to_source = build_v31_source_aliases(store)
    aliases = set(re.findall(r"\bS[0-9]{2,3}\b", markdown))
    unknown_aliases = sorted(aliases - set(alias_to_source))
    if unknown_aliases:
        raise ValueError(f"Unknown source aliases: {unknown_aliases!r}")
    result = markdown
    for alias, source_id in alias_to_source.items():
        result = re.sub(rf"\b{re.escape(alias)}\b", source_id, result)
    return result


def remap_claim_like_source_ids(
    markdown: str, store: EvidenceStore
) -> tuple[str, dict[str, list[str]]]:
    known_source_ids = set(store.sources)
    exact_ids = set(V31_EXACT_SOURCE_ID_PATTERN.findall(markdown))
    remapped: dict[str, list[str]] = {}
    result = markdown
    for source_like_id in sorted(exact_ids - known_source_ids):
        claim_id = f"clm_{source_like_id.removeprefix('src_')}"
        claim = store.claims.get(claim_id)
        if claim is None:
            continue
        supported_source_ids = [
            source_id
            for source_id in claim.source_ids
            if source_id in known_source_ids
        ]
        if not supported_source_ids:
            continue
        result = result.replace(
            source_like_id, ", ".join(supported_source_ids)
        )
        remapped[source_like_id] = supported_source_ids
    return result, remapped


def canonicalize_v31_references(
    revision: V31ReportRevision,
    store: EvidenceStore,
) -> V31ReportRevision:
    materialized_markdown = materialize_v31_source_aliases(
        revision.markdown, store
    )
    remapped_markdown, _ = remap_claim_like_source_ids(
        materialized_markdown, store
    )
    revision = revision.model_copy(update={"markdown": remapped_markdown})
    malformed = find_malformed_source_tokens(revision.markdown)
    if malformed:
        raise ValueError(f"Malformed source tokens: {malformed!r}")

    body, _ = split_v31_report(revision.markdown)
    cited_source_ids = sorted(set(V31_EXACT_SOURCE_ID_PATTERN.findall(body)))
    unknown_source_ids = sorted(set(cited_source_ids) - set(store.sources))
    if unknown_source_ids:
        raise ValueError(f"Unknown source IDs: {unknown_source_ids!r}")
    if not cited_source_ids:
        raise ValueError("The report body contains no stable source citations.")

    reference_lines = [
        f"- [{source_id}] {store.sources[source_id].title} — "
        f"{store.sources[source_id].url}"
        for source_id in cited_source_ids
    ]
    known_claim_ids = set(store.claims)
    used_claim_ids = sorted(
        set(revision.used_claim_ids) & known_claim_ids
    )
    markdown = body + "\n\n## References\n\n" + "\n".join(reference_lines)
    return revision.model_copy(
        update={
            "markdown": markdown,
            "used_source_ids": cited_source_ids,
            "used_claim_ids": used_claim_ids,
        }
    )


def check_v31_report_contract(
    markdown: str,
    case: EvalCase,
    store: EvidenceStore,
) -> V31ContractCheck:
    expectations = case.expectations
    word_count = count_words(markdown)
    lowered_report = markdown.lower()
    malformed_source_tokens = find_malformed_source_tokens(markdown)
    exact_source_ids = sorted(
        set(V31_EXACT_SOURCE_ID_PATTERN.findall(markdown))
    )
    unknown_source_ids = sorted(set(exact_source_ids) - set(store.sources))
    body, references = split_v31_report(markdown)
    body_source_ids = sorted(
        set(V31_EXACT_SOURCE_ID_PATTERN.findall(body))
    )
    reference_source_ids = set(
        V31_EXACT_SOURCE_ID_PATTERN.findall(references)
    )
    missing_reference_ids = sorted(
        set(body_source_ids) - reference_source_ids
    )
    missing_reference_urls = sorted(
        source_id
        for source_id in set(body_source_ids)
        & reference_source_ids
        & set(store.sources)
        if store.sources[source_id].url not in references
        and store.sources[source_id].canonical_url not in references
    )
    missing_sections = [
        section
        for section in expectations.required_sections
        if not re.search(
            rf"^##\s+{re.escape(section)}\s*$",
            markdown,
            re.MULTILINE | re.IGNORECASE,
        )
    ]
    missing_concepts = [
        concept
        for concept in expectations.required_concepts
        if concept.lower() not in lowered_report
    ]

    issues: list[str] = []
    if not expectations.min_words <= word_count <= expectations.max_words:
        issues.append(
            f"Word count {word_count} is outside "
            f"[{expectations.min_words}, {expectations.max_words}]."
        )
    if missing_sections:
        issues.append(f"Missing exact sections: {missing_sections!r}")
    if missing_concepts:
        issues.append(f"Missing concepts: {missing_concepts!r}")
    if len(body_source_ids) < expectations.min_sources:
        issues.append(
            f"Body cites {len(body_source_ids)} sources; "
            f"at least {expectations.min_sources} are required."
        )
    if malformed_source_tokens:
        issues.append(
            f"Malformed source tokens: {malformed_source_tokens!r}"
        )
    if unknown_source_ids:
        issues.append(f"Unknown source IDs: {unknown_source_ids!r}")
    if missing_reference_ids:
        issues.append(
            f"Body citations missing from References: {missing_reference_ids!r}"
        )
    if missing_reference_urls:
        issues.append(
            f"Reference entries missing stored URLs: {missing_reference_urls!r}"
        )

    return V31ContractCheck(
        passed=not issues,
        issues=issues,
        word_count=word_count,
        cited_source_ids=body_source_ids,
        malformed_source_tokens=malformed_source_tokens,
        unknown_source_ids=unknown_source_ids,
        missing_sections=missing_sections,
    )


malformed_guard_example = find_malformed_source_tokens(
    "Valid [src_0123456789ab], invalid [src_036?] and [src__CLAIM_c59?]."
)
assert malformed_guard_example == ["src_036?", "src__CLAIM_c59?"]
print(f"Malformed citation guard: {malformed_guard_example}")


Malformed citation guard: ['src_036?', 'src__CLAIM_c59?']


## 10.2 Bounded V3.1 Report Reviser

The reviser receives the original report, the fixed evaluation contract, and the structured
evidence. Its job is narrower than the Writer Agent's job: it must preserve supported
analysis while repairing only measured gaps. Central claims should prefer stronger source
types, weaker evidence must be labeled, and negative or inconclusive findings must remain
visible.

The model can propose a revision, but Python owns acceptance. At most three calls are allowed.
This bounded `propose -> validate -> retry/fail` loop is the harness improvement.


In [22]:
V31_REPORT_REVISION_INSTRUCTIONS = """
You are the final report contract agent for an evidence-aware research workflow.

Revise the supplied report using only the supplied EvidenceStore. Return one complete
Markdown report and the exact source and claim identifiers used. Obey every contract field
in the payload.

Requirements:
- Use each required section as an exact level-two Markdown heading.
- Keep the total report, including References, inside the requested word-count range.
- Cite factual claims only with allowed source aliases such as [S01] or [S01, S02].
- Never write src_ or clm_ identifiers. Python owns stable identifiers and will replace
  valid S-aliases after your response.
- Never emit placeholder, shortened, invented, or question-mark source aliases.
- Prefer peer-reviewed and primary sources for central claims. Clearly label preprints,
  engineering reports, repositories, and other weaker evidence instead of treating all
  sources as equally strong.
- Calibrate conclusions to claim confidence and source limitations. Preserve negative,
  mixed, or inconclusive findings.
- Explain material evidence gaps and benchmark limitations under ## Limitations.
- Do not add facts absent from the supplied claims. Do not add conversational framing.
- Include ## References. Python will deterministically rebuild its entries after validation.

Return only the structured revision.
"""


def revise_v3_report_to_contract(
    *,
    case: EvalCase,
    baseline_report: str,
    store: EvidenceStore,
    recorder: TraceRecorder | None = None,
    model: str = MODEL_NAME,
    max_revision_rounds: int = 3,
) -> tuple[V31ReportRevision, V31ContractCheck]:
    if max_revision_rounds < 1:
        raise ValueError("max_revision_rounds must be positive.")

    current_report = baseline_report
    current_check = check_v31_report_contract(current_report, case, store)
    target_margin = min(100, max(0, (case.expectations.max_words - case.expectations.min_words) // 4))
    target_min = case.expectations.min_words + target_margin
    target_max = case.expectations.max_words - target_margin

    for revision_round in range(1, max_revision_rounds + 1):
        payload = {
            "user_request": case.user_request,
            "contract": case.expectations.model_dump(),
            "preferred_word_target": [target_min, target_max],
            "allowed_source_aliases": sorted(
                build_v31_source_aliases(store)[1]
            ),
            "validation_feedback": current_check.issues,
            "current_report": prepare_v31_alias_report(
                current_report, store
            ),
            "evidence_catalog": build_v31_evidence_catalog(store),
        }
        if recorder is not None:
            recorder.record(
                "model_call",
                agent="report_contract_agent",
                action="revise_report",
                metadata={"revision_round": revision_round},
            )
        response = client.responses.parse(
            model=model,
            instructions=V31_REPORT_REVISION_INSTRUCTIONS,
            input=json.dumps(payload, ensure_ascii=False),
            text_format=V31ReportRevision,
        )
        if recorder is not None:
            record_response_usage(recorder, response)
        revision = response.output_parsed
        if revision is None:
            raise RuntimeError("V3.1 report reviser returned no structured output.")

        try:
            candidate = canonicalize_v31_references(revision, store)
            current_check = check_v31_report_contract(
                candidate.markdown, case, store
            )
        except ValueError as exc:
            candidate = revision
            current_check = V31ContractCheck(
                passed=False,
                issues=[str(exc)],
                word_count=count_words(revision.markdown),
                cited_source_ids=sorted(
                    set(V31_EXACT_SOURCE_ID_PATTERN.findall(revision.markdown))
                ),
                malformed_source_tokens=find_malformed_source_tokens(
                    revision.markdown
                ),
                unknown_source_ids=sorted(
                    set(V31_EXACT_SOURCE_ID_PATTERN.findall(revision.markdown))
                    - set(store.sources)
                ),
                missing_sections=[],
            )

        if current_check.passed:
            return candidate, current_check

        if recorder is not None:
            recorder.record(
                "guard_rejection",
                agent="report_contract_guard",
                action="reject_report",
                metadata={
                    "revision_round": revision_round,
                    "issues": current_check.issues,
                },
            )
        current_report = candidate.markdown

    raise RuntimeError(
        "V3.1 report failed its contract after "
        f"{max_revision_rounds} rounds: {current_check.issues!r}"
    )


## 10.3 Replay Runner and Same-Case Comparison

The replay runner copies the baseline's observable research trajectory, removes its old
terminal report events, and appends the V3.1 revision events. Therefore research tool calls,
workers, sources, and claims remain fixed, while model-call and token deltas expose the
incremental cost of the new guardrail.

`RUN_V31_BEFORE_AFTER` is `False` by default so opening or running the Notebook does not
silently spend API credits. The saved output below comes from one explicit run.


In [23]:
def copy_trace_for_v31_revision(trace: RunTrace) -> TraceRecorder:
    recorder = TraceRecorder()
    retained_events = [
        event
        for event in trace.events
        if event.event_type
        not in {"report_finalized", "run_completed", "run_failed"}
    ]
    recorder.events = [
        event.model_copy(update={"sequence": sequence})
        for sequence, event in enumerate(retained_events, start=1)
    ]
    recorder.model_calls = trace.metrics.model_calls
    recorder.tool_calls = trace.metrics.tool_calls
    recorder.worker_attempts = trace.metrics.worker_attempts
    recorder.worker_failures = trace.metrics.worker_failures
    recorder.guard_rejections = trace.metrics.guard_rejections
    recorder.input_tokens = trace.metrics.input_tokens
    recorder.output_tokens = trace.metrics.output_tokens
    return recorder


def make_v31_replay_runner(
    baseline: EvalRunResult,
    max_revision_rounds: int = 3,
) -> WorkflowRunner:
    async def runner(case: EvalCase) -> RunTrace:
        if case.id != baseline.case_id:
            raise ValueError(
                f"Replay baseline is for {baseline.case_id!r}, not {case.id!r}."
            )
        baseline_trace = baseline.trace
        if baseline_trace.final_report is None:
            raise ValueError("Replay baseline has no final report.")
        if baseline_trace.evidence_store is None:
            raise ValueError("Replay baseline has no EvidenceStore.")

        recorder = copy_trace_for_v31_revision(baseline_trace)
        started_at = time.perf_counter() - baseline_trace.metrics.duration_seconds
        run_id = f"v3_1_{case.id}_{uuid.uuid4().hex[:8]}"
        recorder.record(
            "supervisor_decision",
            agent="evaluation_supervisor",
            action="apply_v31_report_hardening",
        )

        final_report = None
        error = None
        status: RunStatus = "failed"
        try:
            revision, contract_check = await asyncio.to_thread(
                revise_v3_report_to_contract,
                case=case,
                baseline_report=baseline_trace.final_report,
                store=baseline_trace.evidence_store,
                recorder=recorder,
                model=MODEL_NAME,
                max_revision_rounds=max_revision_rounds,
            )
            final_report = revision.markdown
            recorder.record(
                "report_finalized",
                agent="report_contract_agent",
                action="write",
                metadata={
                    "word_count": contract_check.word_count,
                    "cited_sources": len(contract_check.cited_source_ids),
                },
            )
            recorder.record("run_completed")
            status = "completed"
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"
            recorder.record(
                "run_failed", metadata={"error_type": type(exc).__name__}
            )

        return build_run_trace(
            recorder=recorder,
            run_id=run_id,
            workflow_version="v3.1",
            case_id=case.id,
            status=status,
            started_at=started_at,
            final_report=final_report,
            evidence_store=baseline_trace.evidence_store,
            error=error,
        )

    return runner


def render_v31_before_after(
    baseline: EvalRunResult,
    candidate: EvalRunResult,
) -> str:
    baseline_grades = {grade.grader: grade for grade in baseline.grades}
    candidate_grades = {grade.grader: grade for grade in candidate.grades}
    grader_names = list(dict.fromkeys([*baseline_grades, *candidate_grades]))
    before = baseline.trace.metrics
    after = candidate.trace.metrics
    malformed_before = len(
        next(
            (
                grade.details.get("malformed_source_tokens", [])
                for grade in baseline.grades
                if grade.grader == "evidence_provenance"
            ),
            [],
        )
    )
    malformed_after = len(
        next(
            (
                grade.details.get("malformed_source_tokens", [])
                for grade in candidate.grades
                if grade.grader == "evidence_provenance"
            ),
            [],
        )
    )
    lines = [
        "# V4 V3-to-V3.1 Controlled Before/After Evaluation",
        "",
        "Both versions use the same saved research trace and EvidenceStore. V3.1 changes only the final report contract and citation harness.",
        "",
        "| Metric | V3 baseline | V3.1 hardened | Delta |",
        "|---|---:|---:|---:|",
        f"| Passed | {baseline.passed} | {candidate.passed} | — |",
        f"| Overall score | {baseline.overall_score:.3f} | {candidate.overall_score:.3f} | {candidate.overall_score - baseline.overall_score:+.3f} |",
        f"| Report words | {before.report_word_count} | {after.report_word_count} | {after.report_word_count - before.report_word_count:+d} |",
        f"| Malformed source tokens | {malformed_before} | {malformed_after} | {malformed_after - malformed_before:+d} |",
        f"| Guard rejections | {before.guard_rejections} | {after.guard_rejections} | {after.guard_rejections - before.guard_rejections:+d} |",
        f"| Model calls | {before.model_calls} | {after.model_calls} | {after.model_calls - before.model_calls:+d} |",
        f"| Tool calls | {before.tool_calls} | {after.tool_calls} | {after.tool_calls - before.tool_calls:+d} |",
        f"| Input tokens | {before.input_tokens} | {after.input_tokens} | {after.input_tokens - before.input_tokens:+d} |",
        f"| Output tokens | {before.output_tokens} | {after.output_tokens} | {after.output_tokens - before.output_tokens:+d} |",
        f"| Duration (s) | {before.duration_seconds:.1f} | {after.duration_seconds:.1f} | {after.duration_seconds - before.duration_seconds:+.1f} |",
        "",
        "## Grader Changes",
        "",
        "| Grader | V3 pass | V3 score | V3.1 pass | V3.1 score |",
        "|---|---:|---:|---:|---:|",
    ]
    for grader_name in grader_names:
        baseline_grade = baseline_grades.get(grader_name)
        candidate_grade = candidate_grades.get(grader_name)
        lines.append(
            f"| {grader_name} | "
            f"{baseline_grade.passed if baseline_grade else 'N/A'} | "
            f"{baseline_grade.score if baseline_grade else 0.0:.3f} | "
            f"{candidate_grade.passed if candidate_grade else 'N/A'} | "
            f"{candidate_grade.score if candidate_grade else 0.0:.3f} |"
        )
    lines.extend(
        [
            "",
            "## Interpretation",
            "",
            "- Unchanged tool calls, sources, and claims isolate the report-harness change.",
            "- Model-call, token, and latency deltas are the incremental reliability cost.",
            "- A higher score counts as an improvement only when all hard gates pass.",
        ]
    )
    guard_events = [
        event
        for event in candidate.trace.events
        if event.event_type == "guard_rejection"
    ]
    if guard_events:
        lines.extend(["", "## V3.1 Guard Feedback", ""])
        for event in guard_events:
            revision_round = event.metadata.get("revision_round", "?")
            issues = event.metadata.get("issues", [])
            lines.append(
                f"- Round {revision_round}: {'; '.join(issues)}"
            )
    return "\n".join(lines)


RUN_V31_BEFORE_AFTER = False

if RUN_V31_BEFORE_AFTER:
    saved_results = [
        EvalRunResult.model_validate(V3_BASELINE_RESULT_PAYLOAD)
    ]
    baseline_saved = next(
        result for result in saved_results if result.case_id == "clear_comparison"
    )
    comparison_case = next(
        case for case in EVAL_CASES if case.id == baseline_saved.case_id
    )
    baseline_result = regrade_saved_result(comparison_case, baseline_saved)
    v31_result = await evaluate_one_case(
        case=comparison_case,
        runner=make_v31_replay_runner(baseline_result),
        include_semantic_grader=True,
    )
    before_after_results = [baseline_result, v31_result]
    before_after_markdown = render_v31_before_after(
        baseline_result, v31_result
    )
    show("V3 vs. V3.1 Controlled Evaluation", before_after_markdown)


## V3 vs. V3.1 Controlled Evaluation

# V4 V3-to-V3.1 Controlled Before/After Evaluation

Both versions use the same saved research trace and EvidenceStore. V3.1 changes only the final report contract and citation harness.

| Metric | V3 baseline | V3.1 hardened | Delta |
|---|---:|---:|---:|
| Passed | False | True | — |
| Overall score | 0.829 | 0.962 | +0.133 |
| Report words | 2044 | 1777 | -267 |
| Malformed source tokens | 2 | 0 | -2 |
| Guard rejections | 0 | 1 | +1 |
| Model calls | 21 | 23 | +2 |
| Tool calls | 12 | 12 | +0 |
| Input tokens | 64746 | 79050 | +14304 |
| Output tokens | 90948 | 104118 | +13170 |
| Duration (s) | 445.4 | 527.6 | +82.2 |

## Grader Changes

| Grader | V3 pass | V3 score | V3.1 pass | V3.1 score |
|---|---:|---:|---:|---:|
| scoping_behavior | True | 1.000 | True | 1.000 |
| outcome_contract | False | 0.500 | True | 1.000 |
| trajectory | True | 1.000 | True | 1.000 |
| evidence_provenance | False | 0.857 | True | 1.000 |
| semantic_report | False | 0.787 | True | 0.810 |

## Interpretation

- Unchanged tool calls, sources, and claims isolate the report-harness change.
- Model-call, token, and latency deltas are the incremental reliability cost.
- A higher score counts as an improvement only when all hard gates pass.

## V3.1 Guard Feedback

- Round 1: Word count 1842 is outside [1200, 1800].

# 11. Embedded Evaluation Artifacts

These artifacts are stored directly in the Notebook. They use one fixed baseline trace and EvidenceStore so the V3/V3.1 comparison remains reproducible without external report files.


## 11.1 V3 Baseline Report

## Abstract
From 2023 onward, reflection, tool verification, and multi-agent specialization (MAS) have emerged as core reliability patterns for LLM-based systems. This report synthesizes peer‑reviewed and engineering sources (2023+) to compare these patterns along reliability, safety, and engineering practicality, with concrete guidance for software engineers designing production systems. We define each pattern, map its mechanisms to typical failure modes, summarize empirical evidence and benchmarks, and offer design guidelines, including when to adopt, blend, or avoid a given approach. The analysis highlights complementary use (e.g., reflection plus verification or MAS with provenance) and emphasizes observability, governance, and deployment constraints as central engineering concerns. See discussion and references for source-specific evidence. 

## Pattern Definitions and Mechanisms (post-2023)
- Reflection: an iterative self-analysis loop in which the agent re-examines prior reasoning, intermediate results, and potential failure modes to guide subsequent attempts. Reflection is often embedded in frameworks where diverse critics assess failed reasoning and propose corrective strategies, enabling the main Actor to reflect before re-attempting. This mechanism is described in contemporary work on reflective architectures and multi-agent reflection, with evidence of safety and bias mitigation under certain conditions [src_359dc8fe7c33][src_0261f15ed667][src_c59b57261d05].
- Tool verification: a gating/verification layer that checks outputs, claims, or actions against independent evidence, retrieved context, or provenance before presenting results to users. In RAG/evidence-based generation, verification explicitly assesses fidelity to retrieved sources and context, and gate decisions on evidence support [src_0261f15ed667][src_ea237745ca52]. The mechanism emphasizes provenance, traceability, and alignment with external data sources [src_0261f15ed667, src_ea237745ca52].
- Multi-agent specialization (MAS): an architecture where diverse agent roles (e.g., proposer, critic, verifier) and communication topologies (e.g., message passing) enable debates among specialized viewpoints. Society-of-Mind-inspired frameworks and MAD-like debate patterns structure agent personas and interactions to improve reasoning reliability, albeit with coordination overhead [src_c59b57261d05][src_ea6afdae27cf].

Notes: The three patterns are not uniformly validated across domains; evidence often comes from targeted tasks, early-stage benchmarks, or controlled demonstrations. See limitations in each section and unresolved questions below [src_c59b57261d05][src_359dc8fe7c33].

## Comparative Framework (6+ dimensions)
We compare Reflection, Tool Verification, and MAS across: Reliability (correctness, robustness, safety, consistency); Latency/Compute; Tool Integration; Observability/Traceability; Governance/Maintenance; Failure Modes; Domain Applicability; and Implementation Practicality. The table summarizes key takeaways and known tradeoffs drawn from 2023+ literature and industry reports.

| Dimension | Reflection | Tool Verification | Multi-Agent Specialization (MAS) |
|---|---|---|---|
| Reliability (correctness, safety, bias) | Can improve safety and reduce bias in some tasks, but gains are context- and data-dependent; not universally robust without external critique or feedback loops [src_359dc8fe7c33][src_836d8b321be3]. | Enhancement of factuality when outputs are explicitly tied to verified evidence/context; depends on retrieval quality and evidence integrity [src_0261f15ed667]. | Reliability improves via diverse viewpoints and cross-checks; benefits shown on complex tasks but requires governance and can be sensitive to coordination overhead [src_c59b57261d05][src_ea6afdae27cf]. |
| Latency/Compute | Iterative re-analysis adds additional computation; gains depend on prompt design and evaluation; some reports show marginal improvements for certain prompts [src_c59b57261d05]. | Additional verification steps introduce latency proportional to evidence retrieval and provenance checks; tool integration adds orchestration costs [src_0261f15ed667]. | Coordination across agents incurs communication and scheduling overhead; potential latency increases but can be mitigated with hierarchical roles and parallel debates [src_ea6afdae27cf][src_ea6afdae27cf]. |
| Tool Integration / Evidence Provenance | Reflection benefits when paired with external critique but not inherently tied to tools; integration of reflective critics can be standalone or tool-augmented [src_359dc8fe7c33][src_0261f15ed667]. | Core to pattern; requires reliable sources, provenance tracking, and integration with external tools (e.g., executors, code runners) for factual grounding [src_ea237745ca52][src_0261f15ed667]. | Involves coordinating specialized modules/tools; tool coupling and provenance tracing are central to maintain safety and traceability; orchestration complexity is a key risk [src_ea6afdae27cf][src_fd0183268bfe]. |
| Observability / Traceability | Reflective reasoning traces and diagnostics enable observability of thought processes; industry discussions highlight observability as a critical deployment requirement [src_7fcdc9ac55a4][src_036?]. | Provenance and traceability of tool outputs are essential for debugging and governance; benchmarks and traces (ToolEmu, InjecAgent, AgentDojo) illustrate robustness testing [src_ea237745ca52][src_ea237745ca52]. | Observability of inter-agent communication and decision flows is foundational; governance and auditing challenges arise from multi-agent dynamics and data handling [src_ea6afdae27cf][src_152e841d2d3d]. |
| Governance / Maintenance | Requires careful prompt tuning and monitoring; industry coverage notes reproducibility and governance concerns; ongoing maintenance of reflective policies is nontrivial [src_7fcdc9ac55a4]. | Requires robust data provenance, tool reliability, and update practices for external components; governance of cited sources and evidence flows is critical [src_ea237745ca52]. | Governance is central due to role definitions and inter-agent protocols; maintaining lead-time for updates and ensuring alignment across roles is challenging [src_ea6afdae27cf][src_ea6afdae27cf]. |
| Failure Modes | Context- and prompt-sensitive failures; risk of degraded performance if external critique is absent; mixed evidence across domains [src_c59b57261d05]. | Tool mis grounding, noisy retrieval, or biased evidence can introduce new failure modes; integration issues are non-trivial [src_0261f15ed667]. | Coordination failures, conflicting viewpoints, or mis-specified roles can generate cascading errors; early-stage results require cross-domain validation [src_ea6afdae27cf][src__CLAIM_c59?]. |
| Domain Applicability | Task/data distribution dependent; some domains benefit more from reflection; others rely on external feedback or safe guardrails [src_359dc8fe7c33][src_836d8b321be3]. | High value in information-dense, citation-heavy domains (legal, medical) where provenance is crucial; depends on retrieval quality [src_0261f15ed667]. | Complex, multi-domain tasks with high reliability requirements can benefit; governance and observability overhead may be warranted in regulated contexts [src_c59b57261d05][src_ea6afdae27cf]. |
| Practicality / Implementation | Can be implemented as modular critics or meta-prompting; evidence indicates task-dependent gains and practical caution with tuning [src_359dc8fe7c33][src_0261f15ed667]. | Requires tool integration infrastructure, provenance, and verification flows; many systems still exploring practical deployment; latency concerns exist [src_ea237745ca52]. | Requires orchestration infrastructure, inter-agent communication, and governance; early-stage but promising for hard reasoning tasks; scale and observability remain open problems [src_ea6afdae27cf][src_fd0183268bfe]. |

> Note: The above table reflects current post-2023 findings, which show that none of the patterns universally outperforms the others; practical reliability often improves when patterns are combined with explicit provenance/verification and robust observability. Key sources cited reflect both empirical demonstrations and industry perspectives. See specific source notes after each claim in the text for details and limitations.

## Empirical Evidence and Key Findings (2023+)
- Reflection shows potential safety and bias mitigation benefits, but results are context- and task-dependent. Studies report safer or less biased outputs under certain prompting and feedback configurations, while other setups yield mixed or no universal gains [src_359dc8fe7c33][src_152e841d2d3d][src_836d8b321be3]. The Nature paper (2025) provides strong safety signals in controlled settings but emphasizes that gains rely on task context and evaluation design; replication across domains remains an open question [src_152e841d2d3d]. A 2024 arXiv synthesis notes variable gains depending on prompts and evaluation conditions, underscoring the need for external feedback for robust improvements [src_359dc8fe7c33]. A practical limitation is that many reflection benefits are demonstrated in early-stage or domain-specific benchmarks rather than large-scale production deployments [src_c59b57261d05].
- MAS shows promise by enabling diverse viewpoints to detect/correct errors via debates among specialized critics; this approach is framed as scalable and training-free in targeted demonstrations, but broader benchmarking and cross-domain validation are still needed [src_c59b57261d05][src_fd0183268bfe]. The orchestration of roles such as proposer, critic, and verifier is well-documented, with Society-of-Mind-inspired and MAD-style patterns forming the basis of MAS demonstrations . However, MAS deployments face coordination latency and governance challenges that can offset some reliability gains if not carefully engineered [src_ea6afdae27cf].
- Tool verification and provenance are central to improving factuality and reliability when tools and evidence are integrated. Verification in RAG/evidence-based generation centers on checking alignment with retrieved context and sources; gatekeeping helps reduce hallucinations, albeit with dependency on retrieval quality and context completeness [src_0261f15ed667]. Tool-use tracing and provenance frameworks (e.g., ToolEmu, InjecAgent, AgentDojo) are advancing the evaluation of robustness, but findings are still in early stages and heterogeneous across tool ecosystems [src_ea237745ca52][src_ea237745ca52]. Overall, evidence indicates that tool verification is most effective when anchored in robust provenance and well-integrated tool ecosystems .
- Industry and governance considerations emphasize observability and reproducibility concerns when deploying reflection-based methods, highlighting the need for governance frameworks, robust observability, and controlled experimentation before production rollouts [src_7fcdc9ac55a4]. 
- In practice, several sources advocate for complementary use: reflection and verification together, with traces and provenance enabling safer and more robust decision-making. This view is supported by cross‑pattern analyses and governance discussions in Nature 2025 and related papers [src_152e841d2d3d].

## Practical Guidance for Software Engineers
- Pattern selection based on constraints:
  - Data sensitivity and regulatory constraints: MAS with provenance-aware tool paths and traceable decision logs can support governance requirements; however, MAS adds orchestration complexity and latency that may be unsuitable for ultra-low-latency tasks without careful engineering [src_ea6afdae27cf].
  - Latency budgets and compute constraints: Reflection adds compute for re-analysis; MIS patterns that require cross-agent debate incur extra communication and scheduling overhead; a hybrid approach that limits iterations or uses lightweight critics can help meet tight budgets [src_c59b57261d05][src_ea6afdae27cf].
  - Tool integration maturity: If the system relies on external tools for factual grounding, ensure robust integration (standardized interfaces, retry strategies, and provenance hooks) to minimize tool-induced failures; verification should gate outputs against reliable evidence [src_0261f15ed667].
- Hybrid design patterns for production: Consider combining reflection with verification to exploit their complementary strengths; add provenance traces to improve observability and governance. The literature frames reflection and verification as complementary, with traces enabling safer decision making [src_152e841d2d3d]. MAS can be valuable for hard reasoning tasks but requires explicit governance and observability provisions; plan for instrumentation to diagnose inter-agent failures (e.g., TRAIL to localize failures, MAST to identify multi-agent specification issues) as diagnostic aids in complex deployments [src_ea237745ca52][src_ea237745ca52].
- Observability and testing paradigms: Invest in end-to-end observability of thought processes, tool flows, and agent communications. Industry commentary highlights reproducibility and governance as critical to successful deployment of reflection-based methods; consider adopting formal tracing and provenance standards to support debugging and audits [src_7fcdc9ac55a4]. Diagnostic frameworks like TRAIL (traces of agent actions) and MAST (multi-agent failure localization) offer structured approaches to diagnosing and localizing failures in MAS/reflection systems [src_ea237745ca52].
- Evaluation strategy: Use 2023+ benchmarks and domain-specific evaluation to compare patterns. Early-stage work shows mixed gains for reflection; robust evaluation should include safety, bias, factuality, latency, and governance metrics; for MAS, assess inter-agent coordination latency and failure modes across tasks [src_c59b57261d05][src_ea237745ca52][src_c59b57261d05].

## Limitations and Open Questions
- Publication lag and platform heterogeneity limit generalizability: Many post-2023 results come from targeted tasks, and replication across domains remains an open problem [src_c59b57261d05].
- Cross-pattern comparisons are sparse: Direct, replicated, head-to-head empirical comparisons across reflection, verification, and MAS are limited; most studies focus on one pattern in isolation or small-scale benchmarks [src_ea237745ca52].
- Governance and safety frameworks for MAS remain under active development: Observability, auditing, and regulatory alignment are prominent concerns in industry discourse and early research [src_7fcdc9ac55a4][src_152e841d2d3d].
- Generalizability across sensitive domains: Evidence of safety/bias mitigation from reflection is promising but not universal; more work is needed to understand domain-specific constraints and failure modes [src_359dc8fe7c33][src_836d8b321be3].

## Conclusion
From 2023 onward, reflection, tool verification, and MAS each offer distinct reliability affordances for LLM-based systems. Reflection can enhance safety and bias robustness in some contexts but is not universally effective without external critique and careful evaluation; tool verification provides a concrete path to factuality when tools and provenance are robustly integrated; MAS leverages diverse viewpoints to improve reasoning on complex tasks but introduces coordination and governance challenges. In practice, reliability appears strongest when patterns are combined with explicit provenance, observability, and governance, and when engineering tradeoffs are aligned with domain constraints (data sensitivity, latency, compliance) [src_152e841d2d3d][src_0261f15ed667][src_ea6afdae27cf]. Diagnostic frameworks (TRAIL, MAST) and structured debates offer practical aids for diagnosing and localizing failures in multi-agent/reflection systems, supporting safer deployment in production environments [src_ea237745ca52]. Engineers should tailor pattern choice to task requirements, adopt hybrid designs where appropriate, and invest in observability and governance to realize robust, maintainable LLM-based systems.

## References

- [src_0261f15ed667] Instruct-of-Reflection: Enhancing Large Language Models Iterative Reflection Capabilities via Dynamic-Meta Instruction — https://arxiv.org/html/2503.00902v1
- [src_152e841d2d3d] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://www.nature.com/articles/s44387-025-00045-3
- [src_359dc8fe7c33] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://arxiv.org/html/2406.10400v2
- [src_7fcdc9ac55a4] Breaking Down Reflection Tuning: Enhancing LLM Performance with Self-Learning — https://arize.com/blog/breaking-down-reflection-tuning-enhancing-llm-performance-with-self-learning
- [src_836d8b321be3] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://www.semanticscholar.org/paper/Self-Reflection-Makes-Large-Language-Models-Safer%2C-Liu-Aldahoul/47e6256ee02710c7698f49a0360718d561972e40
- [src_c59b57261d05] Multi-Agent Reflexion Improves Reasoning Abilities in LLMs — https://arxiv.org/html/2512.20845v2
- [src_ea237745ca52] From Agent Traces to Trust: Evidence Tracing and Execution Provenance in LLM Agents — https://arxiv.org/html/2606.04990v1
- [src_ea6afdae27cf] The llama 3 herd of models — https://arxiv.org/abs/2407.21783
- [src_fd0183268bfe] LLMs for Multi-Agent Cooperation — https://xue-guang.com/post/llm-marl


## 11.2 V3.1 Hardened Report

## Abstract
From 2023 onward, reliability patterns for LLM-based software agents have coalesced around three core modalities: reflection, tool verification, and multi-agent specialization (MAS). Each pattern offers distinct mechanisms to improve correctness, safety, and robustness, yet none delivers universal gains across all domains. Reflection is an iterative self-analysis loop in which the agent re-examines prior reasoning, intermediate results, and potential failure modes to guide subsequent attempts. In practice, reflection is frequently augmented by diverse critics or external feedback signals that steer the self-correction loop. This pattern has generated notable safety and bias-mitigation signals in several studies, but the gains prove highly task- and data-dependent and often hinge on the presence of critique signals or external coaching to realize robust improvements in production settings [src_c59b57261d05][src_359dc8fe7c33][src_152e841d2d3d][src_0261f15ed667][src_f940fa9ff9b8]. The early Nature-based and arXiv-evaluated results suggest context-sensitive safety benefits and, in some cases, mixed or even degraded performance without external critique; thus, practitioners should treat reflection as a scalable, modular augmentation rather than a universal remedy [src_152e841d2d3d][src_359dc8fe7c33][src_836d8b321be3]. A practical implication is that deployment with reflection should be coupled to strong governance, provenance, and observability to monitor when the loop yields genuine improvements versus spurious corrections [src_7fcdc9ac55a4][src_ea237745ca52]. 

Tool verification introduces a gating layer that checks outputs, claims, or actions against independent evidence, retrieved context, or provenance before results are released to users. When the verification step is tightly integrated with robust provenance and credible tool ecosystems, factuality and fidelity improve substantially; however, the benefits hinge on the quality of retrieval, the reliability of tools, and the maturity of the verification interfaces. Verification frameworks emphasize provenance, traceability, and alignment with external data sources, and are particularly effective in RAG/evidence-based generation contexts where outputs must be anchored to retrieved context or cited sources [src_ea237745ca52][src_0261f15ed667][src_ea237745ca52]. Empirical work on tool tracing—such as execution provenance and tool-use traces—has matured toward diagnosing where failures originate within an agent’s reasoning and tool flow, with benchmarks (e.g., ToolEmu, InjecAgent, AgentDojo) illustrating robustness tests for instruction following and evidence use. Yet these results are still early-stage and heterogeneous across tool ecosystems, and practical deployment demands careful engineering to mitigate latency and tool-induced errors [src_ea237745ca52]. 

MAS architectures organize reasoning around diverse agent roles (e.g., proposer, critic, verifier) and structured communication topologies to enable debates among specialized viewpoints. Such debate-driven, Society-of-Mind–inspired frameworks attempt to surface and rectify errors that single-agent systems might overlook, offering a scalable path toward reliability on hard tasks without requiring large-scale retraining. The core idea—coordinated disagreements among specialized personas—has been demonstrated in targeted tasks and controlled demonstrations, but it incurs orchestration overhead, governance burden, and potential latency if not carefully engineered. The llama 3 herd and related MAS-inspired work illustrate the promise of coordinated, multi-agent reasoning, while also highlighting practical constraints around inter-agent coordination, observability, and maintainability in production settings [src_c59b57261d05][src_ea6afdae27cf][src_fd0183268bfe]. 

Across these three patterns, the literature from 2023 onward consistently argues that no single approach dominates; reliable production systems typically benefit from hybrid designs that combine reflection or MAS with robust verification, traces, and governance hooks. This blended view aligns with governance-focused discussions in industry and peer-reviewed reporting that emphasize observability, provenance, and domain-aware pattern selection given data-sensitivity and latency constraints [src_152e841d2d3d][src_7fcdc9ac55a4][src_ea6afdae27cf]. In short, reliability emerges from thoughtful design that pairs pattern-specific strengths with strong provenance, evaluation, and governance, rather than from wholesale adoption of any one paradigm. 

### Pattern definitions and mechanisms (post-2023)
- Reflection: An iterative self-analysis loop in which the agent re-examines prior reasoning, intermediate results, and potential failure modes to guide subsequent attempts. Reflection is most effective when paired with diverse critics or external feedback signals that guide re-evaluation and correction before re-attempting the main task; this arrangement is central to contemporary reflective architectures and multi-agent reflection studies that emphasize safety and bias mitigation under particular conditions [src_c59b57261d05][src_359dc8fe7c33][src_0261f15ed667].
- Tool verification: A gating/verification layer that checks outputs, claims, or actions against independent evidence, retrieved context, or provenance before presenting results to users. In RAG and evidence-based generation, verification explicitly assesses fidelity to retrieved sources and context, with an emphasis on provenance, traceability, and alignment to external data sources [src_0261f15ed667][src_ea237745ca52]. The mechanism is most effective when tool ecosystems and interfaces provide reliable provenance hooks and standardized evidence models to prevent drift between claimed outputs and supporting data [src_ea237745ca52].
- Multi-agent specialization (MAS): An architecture where diverse agent roles (e.g., proposer, critic, verifier) and communication topologies (e.g., message passing) enable debates among specialized viewpoints. Society-of-Mind–inspired frameworks and MAD-like debate patterns structure agent personas and interactions to surface and correct errors beyond what a single agent can achieve. Evidence from MAS-focused studies indicates potential reliability gains on hard tasks, but coordination latency, governance complexity, and observability challenges remain important tradeoffs [src_c59b57261d05][src_ea6afdae27cf][src_fd0183268bfe].

Notes: The three patterns are not universally validated across domains; evidence is frequently task- and dataset-specific, with mixed or context-limited outcomes in some cases. See Limitations for domain gaps and measurement challenges that complicate cross-pattern generalizations [src_c59b57261d05][src_359dc8fe7c33][src_ea237745ca52].

## Limitations
- Evidence heterogeneity and publication lag: Much post-2023 evidence derives from preprints, conference papers, or industry reports with heterogeneous validation and limited cross-domain replication. While several high-signal claims appear in peer-reviewed venues (e.g., reflections that yield safety improvements under certain conditions), generalizability remains uncertain. This caution is especially relevant for reflection when the external critique signal is absent or misaligned with the task distribution [src_152e841d2d3d][src_359dc8fe7c33][src_836d8b321be3].
- Cross-pattern comparisons are sparse: Direct, replicated head-to-head comparisons across reflection, verification, and MAS are rare. Most studies evaluate a single pattern in isolation or rely on targeted benchmarks, making it difficult to disentangle pattern-specific gains from domain-specific idiosyncrasies. Consequently, robust cross-pattern transferability and domain-agnostic guidelines are not yet established [src_c59b57261d05][src_ea237745ca52].
- Governance, safety, and observability remain active frontiers: Industry and academic discourse emphasize the need for reproducibility, observability of thought processes, and governance mechanisms when deploying reflection- and MAS-based systems. While frameworks like TRAIL (traces of agent actions) and MAST (multi-agent failure localization) offer diagnostic scaffolds, these tools are still maturing and not yet universally adopted across platforms [src_ea237745ca52][src_7fcdc9ac55a4].
- Latency and compute considerations: Reflection and MAS incur additional computation and inter-agent communication, which can increase end-to-end latency and relevance for latency-constrained tasks. The llama 3 herd paper and related MAS work illustrate coordination overheads that can offset reliability gains if not carefully managed through lightweight critics, bounded debates, and structured governance [src_ea6afdae27cf][src_fd0183268bfe].
- Domain sensitivity and data governance: Pattern choice is increasingly seen as domain- and constraint-aware. Governance, data-handling, and regulatory constraints can favor provenance- and logging-heavy designs (often aligning with verification and MAS governance requirements) or push toward simpler, more transparent reflection flows for ultra-sensitive domains [src_152e841d2d3d][src_7fcdc9ac55a4].
- Tool ecosystems and maturity: The benefits of verification depend on tool reliability, provenance integration, and data quality. Heterogeneous tools and noisy data can introduce new failure modes; thus, verification should be designed with drift controls, retry policies, and clear provenance trails to support debugging and audits [src_0261f15ed667][src_ea237745ca52].
- Evidence quality: A substantial portion of the influential work is preprint-based or industry-led, inviting cautious interpretation and a call for broader replication across domains and longer time horizons to confirm durability in production environments [src_c59b57261d05][src_359dc8fe7c33][src_ea237745ca52].

## Practical Guidance for Software Engineers
- Pattern selection under constraints:
  - Data sensitivity and regulatory requirements: MAS with provenance-aware tool paths and traceable decision logs can support governance demands, but MAS introduces orchestration and latency overhead that may be unsuitable for ultra-low-latency tasks unless carefully engineered [src_ea6afdae27cf].
  - Latency budgets and compute constraints: Reflection adds re-analysis cost; MAS incurs cross-agent communication overhead; a hybrid approach that constrains iterations and uses lightweight critics can help meet tight budgets [src_c59b57261d05][src_ea6afdae27cf].
  - Tool integration maturity: For systems relying on external tools for factual grounding, ensure robust interfaces, standardized provenance, and reliable retry semantics to minimize tool-induced errors; verification should gate outputs against credible evidence [src_0261f15ed667][src_ea237745ca52].
- Hybrid design patterns for production: Consider combining reflection with verification to leverage their complementary strengths; add provenance traces to improve observability and governance. The literature frames reflection and verification as complementary, with traces enabling safer decision making; MAS can augment reasoning for hard problems if accompanied by explicit governance and inter-agent diagnostics [src_152e841d2d3d][src_ea237745ca52][src_7fcdc9ac55a4].
- Observability and testing: Invest in end-to-end observability of thought processes, tool flows, and agent communications. Diagnostic frameworks like TRAIL and MAST can guide debugging and localization of failures in reflection/MAS systems; standardized provenance practices support auditing and compliance requirements [src_ea237745ca52][src_7fcdc9ac55a4].
- Evaluation strategy: Use 2023+ benchmarks and domain-specific tasks to compare patterns, including safety, bias, factuality, latency, and governance metrics. For MAS, assess inter-agent coordination latency and failure modes across tasks; for reflection, quantify safety and bias benefits under varying external feedback conditions [src_c59b57261d05][src_ea237745ca52][src_f940fa9ff9b8].

## Conclusion
From 2023 onward, reflection, tool verification, and MAS each afford distinct reliability affordances for LLM-based systems. Reflection can improve safety and bias robustness in many contexts but is not universally effective without external critique and careful evaluation; tool verification offers a practical route to factuality when provenance and tool ecosystems are robust, though integration effort and latency are nontrivial; MAS can yield higher reliability on complex tasks via diverse viewpoints but requires disciplined governance and observability to manage coordination overhead. The strongest current practical guidance favors hybrid designs that couple pattern-specific strengths with explicit provenance, observability, and governance, while respecting domain constraints such as data sensitivity and latency budgets. Diagnostic frameworks (e.g., TRAIL, MAST) and structured debates offer concrete aids for diagnosing and localizing failures in reflective and multi-agent systems, supporting safer deployment in production environments. Software engineers should tailor pattern choice to task requirements, embrace hybrid designs where appropriate, and invest in robust observability and governance to achieve reliable, maintainable LLM-based systems.

## References

- [src_0261f15ed667] Instruct-of-Reflection: Enhancing Large Language Models Iterative Reflection Capabilities via Dynamic-Meta Instruction — https://arxiv.org/html/2503.00902v1
- [src_152e841d2d3d] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://www.nature.com/articles/s44387-025-00045-3
- [src_359dc8fe7c33] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://arxiv.org/html/2406.10400v2
- [src_7fcdc9ac55a4] Breaking Down Reflection Tuning: Enhancing LLM Performance with Self-Learning — https://arize.com/blog/breaking-down-reflection-tuning-enhancing-llm-performance-with-self-learning
- [src_836d8b321be3] Self-Reflection Makes Large Language Models Safer, Less Biased, and Ideologically Neutral — https://www.semanticscholar.org/paper/Self-Reflection-Makes-Large-Language-Models-Safer%2C-Liu-Aldahoul/47e6256ee02710c7698f49a0360718d561972e40
- [src_c59b57261d05] Multi-Agent Reflexion Improves Reasoning Abilities in LLMs — https://arxiv.org/html/2512.20845v2
- [src_ea237745ca52] From Agent Traces to Trust: Evidence Tracing and Execution Provenance in LLM Agents — https://arxiv.org/html/2606.04990v1
- [src_ea6afdae27cf] The llama 3 herd of models — https://arxiv.org/abs/2407.21783
- [src_f940fa9ff9b8] Self-Reflection in LLM Agents: Effects on Problem-Solving Performance — https://arxiv.org/pdf/2405.06682
- [src_fd0183268bfe] LLMs for Multi-Agent Cooperation — https://xue-guang.com/post/llm-marl


## 11.3 Controlled Before/After Result

# V4 V3-to-V3.1 Controlled Before/After Evaluation

Both versions use the same saved research trace and EvidenceStore. V3.1 changes only the final report contract and citation harness.

| Metric | V3 baseline | V3.1 hardened | Delta |
|---|---:|---:|---:|
| Passed | False | True | — |
| Overall score | 0.829 | 0.962 | +0.133 |
| Report words | 2044 | 1777 | -267 |
| Malformed source tokens | 2 | 0 | -2 |
| Guard rejections | 0 | 1 | +1 |
| Model calls | 21 | 23 | +2 |
| Tool calls | 12 | 12 | +0 |
| Input tokens | 64746 | 79050 | +14304 |
| Output tokens | 90948 | 104118 | +13170 |
| Duration (s) | 445.4 | 527.6 | +82.2 |

## Grader Changes

| Grader | V3 pass | V3 score | V3.1 pass | V3.1 score |
|---|---:|---:|---:|---:|
| scoping_behavior | True | 1.000 | True | 1.000 |
| outcome_contract | False | 0.500 | True | 1.000 |
| trajectory | True | 1.000 | True | 1.000 |
| evidence_provenance | False | 0.857 | True | 1.000 |
| semantic_report | False | 0.787 | True | 0.810 |

## Interpretation

- Unchanged tool calls, sources, and claims isolate the report-harness change.
- Model-call, token, and latency deltas are the incremental reliability cost.
- A higher score counts as an improvement only when all hard gates pass.

## V3.1 Guard Feedback

- Round 1: Word count 1842 is outside [1200, 1800].


# 12. V4 Completion Summary

V4 establishes the evaluation and improvement layer:

- `RunEvent`, `RunMetrics`, and `RunTrace` describe observable executions;
- fixed `EvalCase` objects make regressions repeatable;
- outcome, trajectory, provenance, and semantic graders isolate failure classes;
- workflow adapters expose implementations through one evaluation contract;
- V3.1 turns measured failures into a bounded report-contract revision;
- the controlled V3/V3.1 replay compares quality and incremental reliability cost while
  holding research evidence constant.

V3 is still the clean evidence-aware workflow to study independently. V4 now contains the
improved V3.1 candidate because its design is explicitly derived from evaluation feedback.
The final Python application should move tracing and the V3.1 finalizer into the orchestration
package, add pricing-aware accounting, and run the deterministic evaluation suite in CI.
